
# Kuroshio Daily FINAL R3
## 严格因果分块交叉验证 + 四类审稿基线 + 全套不确定性估计

本 Notebook 面向当前大修中三组核心补充实验：

### A. 时间泛化验证
当前 daily 数据覆盖约 2023-09 ~ 2025-03。由于仍不足以构造真正意义上的“多年滚动后报”，
本轮正式采用审稿人明确允许的另一方案：

```text
strict causal blocked cross-validation / blocked hindcast
```

保持原任务语义：

```text
180-day history
→ +30 d / +60 d / +90 d
```

每个 fold 强制：

```text
all training target dates
<
validation issue start - purge

all validation target dates
<
test issue start - purge
```

因此测试起报时不会使用任何“未来标签”。

### B. Reviewer baselines
基线比较只保留审稿人点名的四类：

```text
Persistence
Climatology
Autoregressive EOF (recursive AR-EOF)
Parameter-matched LSTM
```

与论文主模型：

```text
EOF-TCN-LSTM
```

进行主图比较。

其中 LSTM 不再使用任意隐藏维度，而是**自动匹配 EOF-TCN-LSTM 的参数量**，
使其成为真正的同等规模神经网络基线。

### C. Uncertainty experiments
自动补充：

1. 黑潮轴线识别不确定性：基于 50-m velocity ridge 的局地轴定位宽度；
2. 趋势分析不确定性：moving-block bootstrap 95% CI；
3. 断面指标不确定性：A-E section metrics 的 block-bootstrap 95% CI；
4. 预测误差不确定性：时间块 + 随机种子的 hierarchical moving-block bootstrap；
5. 不同网络结构差异：paired moving-block bootstrap，对网络误差差异给出 95% CI。

### 论文绘图风格
图件采用原稿的克制型 MATLAB/海洋学风格：

- 白色背景；
- 四边黑色坐标轴、向内刻度；
- 浅灰虚线/点线网格；
- 蓝色与橙色为主的低饱和度线条；
- 细线、圆形标记、无装饰性背景；
- 中英文各一套；
- PNG 300 dpi + PDF vector；
- 中文字体预检，缺字体时直接停止，避免乱码。

所有新结果写到：

```text
FINAL_RETRAIN_REVIEWER_R3/
```

不会覆盖此前实验。


In [ ]:

# ===== 0. 配置 =====
from pathlib import Path
from dataclasses import dataclass, asdict
import os, json, math, time, random, warnings, hashlib
import numpy as np
import pandas as pd

@dataclass
class CFG:
    PROJECT_ROOT: str = "/mnt/d/KuroshioDailyR1"

    FINAL_DATASET: str = "R1_FINAL_MANUAL22/dataset/daily_dual_branch_paths_FINAL.npz"
    FINAL_META: str = "R1_FINAL_MANUAL22/dataset/daily_dual_branch_meta_FINAL.csv"

    OUT_ROOT: str = "FINAL_RETRAIN_REVIEWER_R3"
    DATA_DIR: str = "FINAL_RETRAIN_REVIEWER_R3/实验数据"
    CKPT_DIR: str = "FINAL_RETRAIN_REVIEWER_R3/模型权重"
    FIG_ZH_DIR: str = "FINAL_RETRAIN_REVIEWER_R3/图表/中文"
    FIG_EN_DIR: str = "FINAL_RETRAIN_REVIEWER_R3/图表/英文"
    REPORT_DIR: str = "FINAL_RETRAIN_REVIEWER_R3/报告"
    PRED_DIR: str = "FINAL_RETRAIN_REVIEWER_R3/预测结果"

    # 保持原论文6个月历史 -> 未来3个月的任务语义
    LOOKBACK_DAYS: int = 180
    LEAD_DAYS: tuple = (30, 60, 90)

    # 输入允许极短原始缺日；forecast targets禁止插值
    MAX_INPUT_GAP_DAYS: int = 2
    MAX_INPUT_MISSING_FRAC: float = 0.03

    # 原模型路径表示
    LEGACY_BRANCH1_31PTS: bool = True
    BRANCH1_DROP_FIRST: int = 12
    BRANCH2_MAX_LON: float = 145.0

    # blocked temporal CV
    N_FOLDS: int = 3
    # FIX2严格hindcast：显式定义连续、不重叠的 validation/test issue blocks
    STRICT_VAL_BLOCKS: tuple = (
        ("2024-07-20", "2024-08-02"),
        ("2024-08-03", "2024-08-16"),
        ("2024-08-17", "2024-08-30"),
    )
    STRICT_TEST_BLOCKS: tuple = (
        ("2024-11-08", "2024-11-21"),
        ("2024-11-22", "2024-12-05"),
        ("2024-12-06", "2024-12-25"),
    )

    VAL_BLOCK_DAYS: int = 14
    PURGE_DAYS: int = 7
    MIN_TRAIN_ANCHORS: int = 40
    MIN_VAL_ANCHORS: int = 12
    MIN_TEST_ANCHORS: int = 12

    # EOF
    EOF_EXPLAINED_TARGET: float = 0.99
    BRANCH1_MAX_MODES: int = 31
    BRANCH2_MAX_MODES: int = 277  # FIX2: 允许daily数据自动达到99%解释方差

    # AR-EOF
    AR_LAGS: int = 7
    AR_ALPHA_CANDIDATES: tuple = (0.01, 0.1, 1.0, 10.0, 100.0)

    # neural training
    SEEDS: tuple = (20, 21, 22, 23, 24)
    BATCH_SIZE: int = 32
    LR: float = 1e-3
    WEIGHT_DECAY: float = 1e-4
    MAX_EPOCHS_B1: int = 150
    MAX_EPOCHS_B2: int = 100
    EARLY_STOP_PATIENCE: int = 20
    LR_PATIENCE: int = 8
    LR_FACTOR: float = 0.5
    DROPOUT: float = 0.30

    # 运行控制
    FORCE_RETRAIN: bool = False
    TRAIN_FINAL_FROZEN_MODEL: bool = True
    FINAL_VAL_DAYS: int = 35
    USE_AMP: bool = True

    # 绘图
    FIG_DPI: int = 300
    REQUIRE_CJK_FONT: bool = True

    RANDOM_SEED: int = 20260926

    # R3: 主模型与基线
    PRIMARY_MODEL: str = "EOF-TCN-LSTM"
    AR_LAG_CANDIDATES: tuple = (3, 7, 14)
    AR_ALPHA_CANDIDATES_R3: tuple = (0.01, 0.1, 1.0, 10.0)
    PARAM_MATCH_TOLERANCE: float = 0.08

    # R3: 不确定性
    BOOTSTRAP_N: int = 2000
    BOOTSTRAP_SEED: int = 20260928
    PRED_BOOT_BLOCK_DAYS: int = 14
    AXIS_BOOT_BLOCK_DAYS: int = 7
    MONTHLY_BOOT_BLOCK_MONTHS: int = 12

    # 黑潮轴定位不确定性
    FINAL_TRACK_DIR: str = "R1_FINAL_MANUAL22/tracks_final"
    CACHE50_DIR: str = "cache_50m"
    AXIS_POINT_STRIDE: int = 4
    AXIS_PROFILE_HALF_KM: float = 35.0
    AXIS_PROFILE_N: int = 29
    AXIS_CORE_FRAC: float = 0.95
    AXIS_SECONDARY_PEAK_SEP_KM: float = 10.0
    AXIS_AMBIGUOUS_RATIO: float = 0.90

    # 原稿1993-2024长期趋势 / 断面不确定性输入
    # 若暂时不存在，Notebook会生成中文模板，不会伪造长期结果。
    LONGTERM_SERIES_CSV: str = "长期不确定性输入/1993_2024月尺度监测序列.csv"


cfg = CFG()

ROOT = Path(cfg.PROJECT_ROOT)
DATASET_PATH = ROOT / cfg.FINAL_DATASET
META_PATH = ROOT / cfg.FINAL_META

OUT = ROOT / cfg.OUT_ROOT
DATA_DIR = ROOT / cfg.DATA_DIR
CKPT_DIR = ROOT / cfg.CKPT_DIR
FIG_ZH_DIR = ROOT / cfg.FIG_ZH_DIR
FIG_EN_DIR = ROOT / cfg.FIG_EN_DIR
REPORT_DIR = ROOT / cfg.REPORT_DIR
PRED_DIR = ROOT / cfg.PRED_DIR

FINAL_TRACK_DIR = ROOT / cfg.FINAL_TRACK_DIR
CACHE50_DIR = ROOT / cfg.CACHE50_DIR
LONGTERM_SERIES_CSV = ROOT / cfg.LONGTERM_SERIES_CSV
LONGTERM_SERIES_CSV.parent.mkdir(parents=True, exist_ok=True)


for p in [OUT, DATA_DIR, CKPT_DIR, FIG_ZH_DIR, FIG_EN_DIR, REPORT_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(cfg.RANDOM_SEED)
np.random.seed(cfg.RANDOM_SEED)

print(json.dumps(asdict(cfg), indent=2, ensure_ascii=False))


In [ ]:

# ===== 1. 环境 / GPU / 中文字体预检 =====
import importlib.util, platform, sys

required = [
    "numpy","pandas","scipy","sklearn",
    "matplotlib","torch"
]
missing = [x for x in required if importlib.util.find_spec(x) is None]
if missing:
    raise RuntimeError(
        "缺少依赖: " + ", ".join(missing) +
        "\n请在WSL环境中安装后重启Kernel。"
    )

import torch
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.font_manager import FontProperties

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)
    torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def find_cjk_font():
    # 优先直接使用Windows字体，避免WSL中文乱码
    candidates = [
        "/mnt/c/Windows/Fonts/msyh.ttc",
        "/mnt/c/Windows/Fonts/msyhbd.ttc",
        "/mnt/c/Windows/Fonts/simhei.ttf",
        "/mnt/c/Windows/Fonts/simsun.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc",
        "/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc",
    ]

    for p in candidates:
        if Path(p).exists():
            try:
                font_manager.fontManager.addfont(p)
                name = FontProperties(fname=p).get_name()
                return p, name
            except Exception:
                pass

    # 再扫描系统字体
    for p in font_manager.findSystemFonts():
        try:
            name = FontProperties(fname=p).get_name()
            low = name.lower()
            if any(k in low for k in [
                "noto sans cjk", "microsoft yahei",
                "simhei", "simsun", "wenquanyi"
            ]):
                return p, name
        except Exception:
            pass

    return None, None

CJK_FONT_PATH, CJK_FONT_NAME = find_cjk_font()
print("CJK font:", CJK_FONT_PATH, CJK_FONT_NAME)

if cfg.REQUIRE_CJK_FONT and CJK_FONT_NAME is None:
    raise RuntimeError(
        "没有检测到中文字体，因此停止绘图，避免生成中文乱码。\n"
        "WSL通常可以直接访问Windows字体：/mnt/c/Windows/Fonts/msyh.ttc\n"
        "若该文件不存在，可安装 fonts-noto-cjk 后重启Kernel。"
    )

def use_zh_font():
    matplotlib.rcdefaults()
    if CJK_FONT_NAME:
        matplotlib.rcParams["font.family"] = [CJK_FONT_NAME, "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    matplotlib.rcParams["pdf.fonttype"] = 42
    matplotlib.rcParams["ps.fonttype"] = 42

def use_en_font():
    matplotlib.rcdefaults()
    matplotlib.rcParams["font.family"] = ["DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    matplotlib.rcParams["pdf.fonttype"] = 42
    matplotlib.rcParams["ps.fonttype"] = 42

# 中文字体真正渲染 smoke test
use_zh_font()
fig, ax = plt.subplots(figsize=(4,2))
ax.text(0.5,0.5,"黑潮路径预测：中文字体测试",ha="center",va="center")
ax.axis("off")
font_smoke = REPORT_DIR / "中文字体渲染测试.png"
fig.savefig(font_smoke,dpi=140,bbox_inches="tight")
plt.close(fig)
print("中文字体渲染测试 ->", font_smoke)

print("tabulate optional:", importlib.util.find_spec("tabulate") is not None)


# ===== R3 原稿风格绘图参数 =====
# 低饱和度色系参考原稿 Fig.3-Fig.5
STYLE_COLORS = {
    "blue": "#4F7F9D",
    "orange": "#C88463",
    "pale_blue": "#9CB9CA",
    "pale_orange": "#DDB19F",
    "purple_gray": "#B7A6B3",
    "green": "#74B58A",
    "gray": "#6F6F6F",
    "light_gray": "#D9D9D9",
    "dark": "#222222",
}

def manuscript_style(lang="en"):
    matplotlib.rcdefaults()
    if lang == "zh":
        if CJK_FONT_NAME:
            matplotlib.rcParams["font.family"] = [CJK_FONT_NAME, "DejaVu Sans"]
    else:
        # 原稿整体接近Times系论文图风格；WSL若无Times自动回退DejaVu Serif
        serif = []
        for name in ["Times New Roman", "Liberation Serif", "DejaVu Serif"]:
            serif.append(name)
        matplotlib.rcParams["font.family"] = "serif"
        matplotlib.rcParams["font.serif"] = serif

    matplotlib.rcParams.update({
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": STYLE_COLORS["dark"],
        "axes.linewidth": 0.8,
        "axes.grid": True,
        "grid.color": STYLE_COLORS["light_gray"],
        "grid.linestyle": ":",
        "grid.linewidth": 0.6,
        "grid.alpha": 0.75,
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.top": True,
        "ytick.right": True,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,
        "font.size": 10,
        "axes.labelsize": 11,
        "axes.titlesize": 11,
        "legend.fontsize": 9,
        "legend.frameon": False,
        "lines.linewidth": 1.6,
        "lines.markersize": 4.5,
        "axes.unicode_minus": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })

def save_dual_language(builder, zh_stem, en_stem):
    manuscript_style("zh")
    fig = builder("zh")
    fig.savefig(FIG_ZH_DIR/(zh_stem+".png"), dpi=cfg.FIG_DPI, bbox_inches="tight")
    fig.savefig(FIG_ZH_DIR/(zh_stem+".pdf"), bbox_inches="tight")
    plt.close(fig)

    manuscript_style("en")
    fig = builder("en")
    fig.savefig(FIG_EN_DIR/(en_stem+".png"), dpi=cfg.FIG_DPI, bbox_inches="tight")
    fig.savefig(FIG_EN_DIR/(en_stem+".pdf"), bbox_inches="tight")
    plt.close(fig)


In [ ]:

# ===== 2. 读取最终人工QC后的日尺度数据 =====
if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"未找到最终冻结数据集:\n{DATASET_PATH}\n"
        "请先在 MANUAL22 Notebook 中运行“冻结最终轨迹”和“生成FINAL双分支日尺度数据集”两个单元。"
    )

with np.load(DATASET_PATH, allow_pickle=False) as z:
    dates = pd.to_datetime(z["dates"])
    B1 = np.asarray(z["branch1_lon"], dtype=np.float32)
    LAT1 = np.asarray(z["branch1_target_lats"], dtype=np.float64)
    B2 = np.asarray(z["branch2_lat"], dtype=np.float32)
    LON2 = np.asarray(z["branch2_target_lons"], dtype=np.float64)
    src = np.asarray(z["source"]).astype(str) if "source" in z.files else np.array(["unknown"]*len(dates))

order = np.argsort(dates)
dates = dates[order]
B1 = B1[order]
B2 = B2[order]
src = src[order]

# 去重
dup = pd.Index(dates).duplicated(keep="last")
if dup.any():
    keep = ~dup
    dates, B1, B2, src = dates[keep], B1[keep], B2[keep], src[keep]

# 忠实保持原Branch1 31点表示
if cfg.LEGACY_BRANCH1_31PTS:
    if B1.shape[1] <= cfg.BRANCH1_DROP_FIRST:
        raise RuntimeError("Branch1维度不足，无法执行legacy 31-point crop")
    B1 = B1[:, cfg.BRANCH1_DROP_FIRST:]
    LAT1 = LAT1[cfg.BRANCH1_DROP_FIRST:]

# Branch2使用122~145E完整范围
mask2 = LON2 <= cfg.BRANCH2_MAX_LON + 1e-9
B2 = B2[:, mask2]
LON2 = LON2[mask2]

if np.isnan(B1).any() or np.isnan(B2).any():
    raise RuntimeError("FINAL dataset 内部仍有 NaN。请先修复冻结数据集。")

calendar_days = int((dates.max()-dates.min()).days+1)
coverage = len(dates)/calendar_days

print("date range:",dates.min(),"->",dates.max())
print("daily path N:",len(dates))
print("calendar coverage:",f"{coverage:.3%}")
print("Branch1:",B1.shape,"lat:",LAT1[0],"->",LAT1[-1])
print("Branch2:",B2.shape,"lon:",LON2[0],"->",LON2[-1])

audit = pd.DataFrame({
    "日期":dates,
    "来源":src,
})
audit.to_csv(
    DATA_DIR/"00_最终日尺度路径数据审计.csv",
    index=False, encoding="utf-8-sig"
)

source_counts = audit["来源"].value_counts().rename_axis("来源").reset_index(name="天数")
source_counts.to_csv(
    DATA_DIR/"00_路径来源统计.csv",
    index=False, encoding="utf-8-sig"
)
display(source_counts)


In [ ]:

# ===== 3. 构建180-day → +30/+60/+90 day anchors =====
full_dates = pd.date_range(dates.min(), dates.max(), freq="D")

def matrix_to_daily_df(mat):
    df = pd.DataFrame(
        mat,
        index=pd.DatetimeIndex(dates).normalize()
    )
    return df.reindex(full_dates)

df1 = matrix_to_daily_df(B1)
df2 = matrix_to_daily_df(B2)

present = pd.Series(False,index=full_dates)
present.loc[pd.DatetimeIndex(dates).normalize()] = True

def longest_missing_run(mask):
    best=cur=0
    for v in mask:
        if bool(v):
            cur+=1
            best=max(best,cur)
        else:
            cur=0
    return best

def get_input_window(df,issue):
    start = issue-pd.Timedelta(days=cfg.LOOKBACK_DAYS-1)
    win = df.loc[start:issue].copy()

    if len(win) != cfg.LOOKBACK_DAYS:
        return None

    row_missing = win.isna().any(axis=1).to_numpy()

    if row_missing.mean() > cfg.MAX_INPUT_MISSING_FRAC:
        return None
    if longest_missing_run(row_missing) > cfg.MAX_INPUT_GAP_DAYS:
        return None

    win = win.interpolate(
        method="time",
        limit=cfg.MAX_INPUT_GAP_DAYS,
        limit_area="inside"
    ).ffill(
        limit=cfg.MAX_INPUT_GAP_DAYS
    ).bfill(
        limit=cfg.MAX_INPUT_GAP_DAYS
    )

    if win.isna().any().any():
        return None

    return win.to_numpy(dtype=np.float32)

def actual_target(df,dt):
    if dt not in df.index:
        return None
    if not bool(present.loc[dt]):
        return None
    row = df.loc[dt]
    if row.isna().any():
        return None
    return row.to_numpy(dtype=np.float32)

anchors=[]

min_issue = full_dates.min()+pd.Timedelta(days=cfg.LOOKBACK_DAYS-1)
max_issue = full_dates.max()-pd.Timedelta(days=max(cfg.LEAD_DAYS))

for issue in pd.date_range(min_issue,max_issue,freq="D"):
    x1=get_input_window(df1,issue)
    x2=get_input_window(df2,issue)
    if x1 is None or x2 is None:
        continue

    tdates=[issue+pd.Timedelta(days=d) for d in cfg.LEAD_DAYS]
    y1=[actual_target(df1,d) for d in tdates]
    y2=[actual_target(df2,d) for d in tdates]

    if any(v is None for v in y1) or any(v is None for v in y2):
        continue

    anchors.append({
        "issue":issue,
        "target_dates":tdates,
        "target_max":max(tdates),
        "X1":x1,
        "X2":x2,
        "Y1":np.stack(y1).astype(np.float32),
        "Y2":np.stack(y2).astype(np.float32),
    })

print("valid 180-day anchors:",len(anchors))

if len(anchors) < 180:
    warnings.warn(
        f"180-day anchors只有 {len(anchors)} 个。"
        "脚本仍可运行，但blocked-CV稳定性会下降。"
    )

anchor_index = pd.DataFrame({
    "样本编号":np.arange(len(anchors)),
    "起报日期":[a["issue"] for a in anchors],
    "30天目标日期":[a["target_dates"][0] for a in anchors],
    "60天目标日期":[a["target_dates"][1] for a in anchors],
    "90天目标日期":[a["target_dates"][2] for a in anchors],
    "最晚目标日期":[a["target_max"] for a in anchors],
})
anchor_index.to_csv(
    DATA_DIR/"01_180天历史预测样本索引.csv",
    index=False, encoding="utf-8-sig"
)
display(anchor_index.head())


In [ ]:

# ===== 4. FIX2严格因果 3-fold blocked hindcast =====
# 原则：
# 1) training 的所有 target_max 必须早于 validation issue start - purge
# 2) validation 的所有 target_max 必须早于 test issue start - purge
# 3) test 以 forecast issue date 划分，而不是以 target_max 划分
# 4) 三个 test issue blocks 连续向后推进且互不重叠

if len(cfg.STRICT_VAL_BLOCKS) != cfg.N_FOLDS or len(cfg.STRICT_TEST_BLOCKS) != cfg.N_FOLDS:
    raise RuntimeError("STRICT_VAL_BLOCKS / STRICT_TEST_BLOCKS 与 N_FOLDS 不一致")

FOLDS=[]
fold_rows=[]
sample_rows=[]
leakage_rows=[]

for fi in range(cfg.N_FOLDS):
    val_start=pd.Timestamp(cfg.STRICT_VAL_BLOCKS[fi][0])
    val_end=pd.Timestamp(cfg.STRICT_VAL_BLOCKS[fi][1])

    test_start=pd.Timestamp(cfg.STRICT_TEST_BLOCKS[fi][0])
    test_end=pd.Timestamp(cfg.STRICT_TEST_BLOCKS[fi][1])

    # 截止条件保证所有训练标签在validation起报前已经可用
    train_target_cutoff = val_start - pd.Timedelta(days=cfg.PURGE_DAYS)

    # validation本身按issue date选取；其全部标签必须在test起报前可用
    val_target_cutoff = test_start - pd.Timedelta(days=cfg.PURGE_DAYS)

    tr=[]
    va=[]
    te=[]

    for i,a in enumerate(anchors):
        issue=pd.Timestamp(a["issue"])
        tmax=pd.Timestamp(a["target_max"])

        if tmax < train_target_cutoff:
            tr.append(i)

        elif (
            val_start <= issue <= val_end and
            tmax < val_target_cutoff
        ):
            va.append(i)

        elif test_start <= issue <= test_end:
            te.append(i)

    tr=np.asarray(tr,dtype=int)
    va=np.asarray(va,dtype=int)
    te=np.asarray(te,dtype=int)

    if len(tr)<cfg.MIN_TRAIN_ANCHORS:
        raise RuntimeError(f"Fold {fi+1} train anchors不足: {len(tr)}")
    if len(va)<cfg.MIN_VAL_ANCHORS:
        raise RuntimeError(f"Fold {fi+1} val anchors不足: {len(va)}")
    if len(te)<cfg.MIN_TEST_ANCHORS:
        raise RuntimeError(f"Fold {fi+1} test anchors不足: {len(te)}")

    # -------- 严格泄漏审计 --------
    train_target_max=max(pd.Timestamp(anchors[i]["target_max"]) for i in tr)
    val_issue_min=min(pd.Timestamp(anchors[i]["issue"]) for i in va)
    val_target_max=max(pd.Timestamp(anchors[i]["target_max"]) for i in va)
    test_issue_min=min(pd.Timestamp(anchors[i]["issue"]) for i in te)

    check_train_to_val = train_target_max < (val_issue_min-pd.Timedelta(days=cfg.PURGE_DAYS))
    check_val_to_test = val_target_max < (test_issue_min-pd.Timedelta(days=cfg.PURGE_DAYS))

    # 所有实际target日期集合也要求互不重叠
    def target_date_set(indices):
        s=set()
        for j in indices:
            s.update(pd.Timestamp(x) for x in anchors[j]["target_dates"])
        return s

    tr_targets=target_date_set(tr)
    va_targets=target_date_set(va)
    te_targets=target_date_set(te)

    overlap_tr_va=len(tr_targets & va_targets)
    overlap_tr_te=len(tr_targets & te_targets)
    overlap_va_te=len(va_targets & te_targets)

    check_no_target_overlap = (
        overlap_tr_va==0 and
        overlap_tr_te==0 and
        overlap_va_te==0
    )

    if not (check_train_to_val and check_val_to_test and check_no_target_overlap):
        raise RuntimeError(
            f"Fold {fi+1} 时间泄漏审计失败: "
            f"train→val={check_train_to_val}, "
            f"val→test={check_val_to_test}, "
            f"overlap={overlap_tr_va}/{overlap_tr_te}/{overlap_va_te}"
        )

    f={
        "fold":fi+1,
        "train_idx":tr,
        "val_idx":va,
        "test_idx":te,

        "train_target_end":train_target_max,
        "val_issue_start":val_issue_min,
        "val_issue_end":max(pd.Timestamp(anchors[i]["issue"]) for i in va),
        "val_target_end":val_target_max,
        "test_issue_start":test_issue_min,
        "test_issue_end":max(pd.Timestamp(anchors[i]["issue"]) for i in te),
        "test_target_end":max(pd.Timestamp(anchors[i]["target_max"]) for i in te),
    }
    FOLDS.append(f)

    fold_rows.append({
        "折次":fi+1,
        "训练样本数":len(tr),
        "验证样本数":len(va),
        "测试样本数":len(te),
        "训练标签最晚日期":train_target_max,
        "验证起报开始":f["val_issue_start"],
        "验证起报结束":f["val_issue_end"],
        "验证标签最晚日期":val_target_max,
        "测试起报开始":f["test_issue_start"],
        "测试起报结束":f["test_issue_end"],
        "测试标签最晚日期":f["test_target_end"],
        "隔离天数":cfg.PURGE_DAYS,
    })

    leakage_rows.append({
        "折次":fi+1,
        "训练标签最晚日期":train_target_max,
        "验证最早起报日期":val_issue_min,
        "验证标签最晚日期":val_target_max,
        "测试最早起报日期":test_issue_min,
        "训练到验证严格因果":check_train_to_val,
        "验证到测试严格因果":check_val_to_test,
        "训练验证目标日期重叠数":overlap_tr_va,
        "训练测试目标日期重叠数":overlap_tr_te,
        "验证测试目标日期重叠数":overlap_va_te,
        "泄漏审计":"PASS",
    })

    for split,key in [("训练","train_idx"),("验证","val_idx"),("测试","test_idx")]:
        for idx in f[key]:
            sample_rows.append({
                "折次":fi+1,
                "集合":split,
                "样本编号":int(idx),
                "起报日期":anchors[idx]["issue"],
                "30天目标日期":anchors[idx]["target_dates"][0],
                "60天目标日期":anchors[idx]["target_dates"][1],
                "90天目标日期":anchors[idx]["target_dates"][2],
                "最晚目标日期":anchors[idx]["target_max"],
            })

fold_df=pd.DataFrame(fold_rows)
sample_df=pd.DataFrame(sample_rows)
leakage_df=pd.DataFrame(leakage_rows)

fold_df.to_csv(
    DATA_DIR/"02_严格分块后报折次定义.csv",
    index=False,encoding="utf-8-sig"
)
sample_df.to_csv(
    DATA_DIR/"02_严格分块后报逐样本划分.csv",
    index=False,encoding="utf-8-sig"
)
leakage_df.to_csv(
    DATA_DIR/"02_严格时序数据泄漏审计.csv",
    index=False,encoding="utf-8-sig"
)

display(fold_df)
display(leakage_df)

print("[STRICT TEMPORAL LEAKAGE AUDIT] ALL PASS")


In [ ]:

# ===== 5. Fold-specific train-only scaler + EOF =====
from sklearn.linear_model import Ridge

class BranchPreprocessor:
    def __init__(self,max_modes,explained_target=0.99):
        self.max_modes=max_modes
        self.explained_target=explained_target

    def fit(self,X_train):
        # X_train: [N,T,M], 只用train input
        flat=X_train.reshape(-1,X_train.shape[-1]).astype(np.float64)

        self.mu=flat.mean(axis=0)
        self.std=flat.std(axis=0)
        self.std[self.std<1e-6]=1.0

        Z=(flat-self.mu)/self.std
        # SVD EOF
        _,s,vt=np.linalg.svd(Z,full_matrices=False)
        variance=s**2
        ratio=variance/(variance.sum()+1e-12)
        cum=np.cumsum(ratio)

        need=int(np.searchsorted(cum,self.explained_target)+1)
        if self.max_modes is not None:
            need=min(need,self.max_modes)
        need=min(need,vt.shape[0])

        self.V=vt[:need].T.astype(np.float32)
        self.explained=float(cum[need-1])
        self.n_modes=need
        return self

    def standardize(self,A):
        return ((A-self.mu)/self.std).astype(np.float32)

    def destandardize(self,Z):
        return (Z*self.std+self.mu).astype(np.float32)

    def path_to_pc(self,A):
        Z=self.standardize(A)
        return np.matmul(Z,self.V).astype(np.float32)

    def pc_to_path(self,PC):
        Z=np.matmul(PC,self.V.T)
        return self.destandardize(Z)

    def neural_input(self,X):
        Z=self.standardize(X)
        D=np.zeros_like(Z)
        D[:,1:]=Z[:,1:]-Z[:,:-1]
        return np.concatenate([Z,D],axis=-1).astype(np.float32)

def stack_anchor(indices,key):
    return np.stack([anchors[i][key] for i in indices]).astype(np.float32)

def dates_for(indices):
    issues=np.array([anchors[i]["issue"] for i in indices],dtype="datetime64[D]")
    targets=np.array([
        [np.datetime64(d.date()) for d in anchors[i]["target_dates"]]
        for i in indices
    ])
    return issues,targets

def prepare_fold(f):
    tri=f["train_idx"]; vai=f["val_idx"]; tei=f["test_idx"]

    X1tr=stack_anchor(tri,"X1")
    X2tr=stack_anchor(tri,"X2")

    pp1=BranchPreprocessor(
        cfg.BRANCH1_MAX_MODES,
        cfg.EOF_EXPLAINED_TARGET
    ).fit(X1tr)

    pp2=BranchPreprocessor(
        cfg.BRANCH2_MAX_MODES,
        cfg.EOF_EXPLAINED_TARGET
    ).fit(X2tr)

    out={"pp1":pp1,"pp2":pp2}

    for split,idx in [("train",tri),("val",vai),("test",tei)]:
        X1=stack_anchor(idx,"X1")
        X2=stack_anchor(idx,"X2")
        Y1=stack_anchor(idx,"Y1")
        Y2=stack_anchor(idx,"Y2")
        issues,tdates=dates_for(idx)

        out[split]={
            "X1_raw":X1,
            "X2_raw":X2,
            "Y1_raw":Y1,
            "Y2_raw":Y2,

            "X1":pp1.neural_input(X1),
            "X2":pp2.neural_input(X2),

            "X1_pc":pp1.path_to_pc(X1),
            "X2_pc":pp2.path_to_pc(X2),

            "Y1_pc":pp1.path_to_pc(Y1),
            "Y2_pc":pp2.path_to_pc(Y2),

            "issues":issues,
            "target_dates":tdates,
        }

    return out

FOLD_DATA={}
preprocess_rows=[]

for f in FOLDS:
    fd=prepare_fold(f)
    FOLD_DATA[f["fold"]]=fd

    preprocess_rows.append({
        "折次":f["fold"],
        "分支1_EOF模态数":fd["pp1"].n_modes,
        "分支1_累计解释方差":fd["pp1"].explained,
        "分支2_EOF模态数":fd["pp2"].n_modes,
        "分支2_累计解释方差":fd["pp2"].explained,
    })

preprocess_df=pd.DataFrame(preprocess_rows)
preprocess_df.to_csv(
    DATA_DIR/"03_各折训练集EOF配置.csv",
    index=False, encoding="utf-8-sig"
)
display(preprocess_df)

# FIX2: daily正式实验要求两分支都真正达到99% training variance
if (preprocess_df["分支1_累计解释方差"] < cfg.EOF_EXPLAINED_TARGET-1e-6).any():
    raise RuntimeError("Branch1 EOF未达到99%解释方差")
if (preprocess_df["分支2_累计解释方差"] < cfg.EOF_EXPLAINED_TARGET-1e-6).any():
    raise RuntimeError(
        "Branch2 EOF未达到99%解释方差。"
        "请增加 BRANCH2_MAX_MODES；不要继续正式训练。"
    )

print("[EOF 99% TRAIN-ONLY GATE] ALL PASS")


In [ ]:

# ===== 6. 路径合并与公里尺度评估 =====
def merge_branches(b1_lon,b2_lat):
    b1_lon=np.asarray(b1_lon,float)
    b2_lat=np.asarray(b2_lat,float)

    lon1=b1_lon
    lat1=LAT1

    end_lon=float(lon1[-1])
    end_lat=float(lat1[-1])

    valid=np.where(LON2>end_lon)[0]
    if len(valid)==0:
        return lon1.copy(),lat1.copy()

    d=haversine_km(
        end_lon,end_lat,
        LON2[valid],b2_lat[valid]
    )
    k=int(valid[np.argmin(d)])

    lon=np.r_[lon1,LON2[k:]]
    lat=np.r_[lat1,b2_lat[k:]]
    return lon.astype(np.float32),lat.astype(np.float32)

def haversine_km(lon1,lat1,lon2,lat2):
    lon1,lat1,lon2,lat2=map(np.asarray,(lon1,lat1,lon2,lat2))
    R=6371.0088
    p1,p2=np.deg2rad(lat1),np.deg2rad(lat2)
    dl=np.deg2rad(lon2-lon1)
    dp=p2-p1
    aa=np.sin(dp/2)**2+np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*R*np.arcsin(np.minimum(1.0,np.sqrt(aa)))

def pairwise_local_km(lon_a,lat_a,lon_b,lat_b):
    lat0=float(np.nanmean(np.r_[lat_a,lat_b]))
    lon0=float(np.nanmean(np.r_[lon_a,lon_b]))

    def xy(lon,lat):
        x=(np.asarray(lon)-lon0)*111.32*np.cos(np.deg2rad(lat0))
        y=(np.asarray(lat)-lat0)*111.32
        return np.column_stack([x,y])

    A=xy(lon_a,lat_a)
    B=xy(lon_b,lat_b)
    return np.sqrt(((A[:,None,:]-B[None,:,:])**2).sum(axis=2))

def cross_track_metrics(pred_lon,pred_lat,true_lon,true_lat):
    d=pairwise_local_km(pred_lon,pred_lat,true_lon,true_lat)

    # symmetric nearest-path distance
    a=d.min(axis=1)
    b=d.min(axis=0)
    x=np.r_[a,b]

    return {
        "横向误差平均值_km":float(np.mean(x)),
        "横向误差中位数_km":float(np.median(x)),
        "横向误差RMSE_km":float(np.sqrt(np.mean(x**2))),
        "横向误差P90_km":float(np.percentile(x,90)),
        "横向误差P95_km":float(np.percentile(x,95)),
        "横向误差最大值_km":float(np.max(x)),
    }

def evaluate_predictions(
    fold,model,seed,
    pred1,pred2,
    true1,true2,
    issues,target_dates
):
    rows=[]

    for n in range(len(pred1)):
        for j,lead in enumerate(cfg.LEAD_DAYS):
            plon,plat=merge_branches(pred1[n,j],pred2[n,j])
            tlon,tlat=merge_branches(true1[n,j],true2[n,j])

            m=cross_track_metrics(plon,plat,tlon,tlat)

            row={
                "折次":int(fold),
                "模型":model,
                "随机种子":seed,
                "起报日期":pd.Timestamp(issues[n]),
                "目标日期":pd.Timestamp(target_dates[n,j]),
                "提前期_天":int(lead),

                "分支1经度RMSE_度":float(np.sqrt(np.mean((pred1[n,j]-true1[n,j])**2))),
                "分支1经度MAE_度":float(np.mean(np.abs(pred1[n,j]-true1[n,j]))),
                "分支2纬度RMSE_度":float(np.sqrt(np.mean((pred2[n,j]-true2[n,j])**2))),
                "分支2纬度MAE_度":float(np.mean(np.abs(pred2[n,j]-true2[n,j]))),
                **m
            }
            rows.append(row)

    return pd.DataFrame(rows)


In [ ]:

# ===== 7. Reviewer baselines：Persistence / Climatology / Recursive AR-EOF =====
from sklearn.linear_model import Ridge

def persistence_predict(split):
    p1=np.repeat(split["X1_raw"][:,-1:, :],len(cfg.LEAD_DAYS),axis=1)
    p2=np.repeat(split["X2_raw"][:,-1:, :],len(cfg.LEAD_DAYS),axis=1)
    return p1.astype(np.float32),p2.astype(np.float32)

def fit_climatology(fd,branch):
    """
    仅使用训练目标。
    优先月条件均值；当某月训练样本太少时退化为全训练集均值。
    """
    Y=fd["train"][f"Y{branch}_raw"]
    tdates=fd["train"]["target_dates"]
    global_mean=Y.mean(axis=0)
    month_map={}

    for j in range(len(cfg.LEAD_DAYS)):
        months=pd.to_datetime(tdates[:,j]).month.to_numpy()
        for m in range(1,13):
            ids=np.where(months==m)[0]
            if len(ids)>=3:
                month_map[(j,m)]=Y[ids,j].mean(axis=0)

    return {"global":global_mean,"month":month_map}

def climatology_predict(model,target_dates):
    N=len(target_dates)
    L=len(cfg.LEAD_DAYS)
    M=model["global"].shape[-1]
    out=np.zeros((N,L,M),dtype=np.float32)
    for n in range(N):
        for j in range(L):
            month=pd.Timestamp(target_dates[n,j]).month
            out[n,j]=model["month"].get((j,month),model["global"][j])
    return out

# ---------- 真正递推式 AR-EOF ----------
def daily_branch_matrix(branch):
    return df1 if branch==1 else df2

def contiguous_training_pc_series(fd,branch,train_end):
    df=daily_branch_matrix(branch).loc[:pd.Timestamp(train_end)].copy()
    # 仅允许短缺口，规则与预测输入一致
    df=df.interpolate(
        method="time",
        limit=cfg.MAX_INPUT_GAP_DAYS,
        limit_area="inside"
    ).ffill(limit=cfg.MAX_INPUT_GAP_DAYS).bfill(limit=cfg.MAX_INPUT_GAP_DAYS)
    df=df.dropna()

    pp=fd[f"pp{branch}"]
    pc=pp.path_to_pc(df.to_numpy(dtype=np.float32))
    return pd.DatetimeIndex(df.index),pc.astype(np.float32)

def fit_one_step_ar(pc,lags,alpha):
    if len(pc)<=lags+5:
        raise RuntimeError("AR训练序列过短")
    X=[];Y=[]
    for t in range(lags,len(pc)):
        X.append(pc[t-lags:t].reshape(-1))
        Y.append(pc[t])
    X=np.asarray(X,np.float32)
    Y=np.asarray(Y,np.float32)
    reg=Ridge(alpha=float(alpha))
    reg.fit(X,Y)
    return reg

def observed_pc_history(fd,branch,issue,lags):
    df=daily_branch_matrix(branch)
    start=pd.Timestamp(issue)-pd.Timedelta(days=lags-1)
    q=df.loc[start:pd.Timestamp(issue)].copy()
    q=q.interpolate(
        method="time",
        limit=cfg.MAX_INPUT_GAP_DAYS,
        limit_area="inside"
    ).ffill(limit=cfg.MAX_INPUT_GAP_DAYS).bfill(limit=cfg.MAX_INPUT_GAP_DAYS)
    if len(q)!=lags or q.isna().any().any():
        return None
    return fd[f"pp{branch}"].path_to_pc(
        q.to_numpy(dtype=np.float32)
    ).astype(np.float32)

def recursive_ar_pc_forecast(reg,hist_pc,lags,max_lead):
    seq=[x.copy() for x in hist_pc]
    out=[]
    for step in range(1,max_lead+1):
        x=np.asarray(seq[-lags:],np.float32).reshape(1,-1)
        nxt=reg.predict(x)[0].astype(np.float32)
        seq.append(nxt)
        if step in cfg.LEAD_DAYS:
            out.append(nxt.copy())
    return np.stack(out).astype(np.float32)

def fit_recursive_ar_branch(fd,f,branch):
    train_dates,train_pc=contiguous_training_pc_series(
        fd,branch,f["train_target_end"]
    )
    best=None

    for lags in cfg.AR_LAG_CANDIDATES:
        for alpha in cfg.AR_ALPHA_CANDIDATES_R3:
            try:
                reg=fit_one_step_ar(train_pc,lags,alpha)
            except Exception:
                continue

            preds=[]
            truths=[]
            for n,issue in enumerate(fd["val"]["issues"]):
                hist=observed_pc_history(fd,branch,pd.Timestamp(issue),lags)
                if hist is None:
                    continue
                pred=recursive_ar_pc_forecast(reg,hist,lags,max(cfg.LEAD_DAYS))
                preds.append(pred)
                truths.append(fd["val"][f"Y{branch}_pc"][n])

            if len(preds)<max(5,len(fd["val"]["issues"])//2):
                continue

            preds=np.stack(preds)
            truths=np.stack(truths)
            mse=float(np.mean((preds-truths)**2))

            if best is None or mse<best["val_mse"]:
                best={
                    "lags":int(lags),
                    "alpha":float(alpha),
                    "val_mse":mse,
                    "model":reg,
                }

    if best is None:
        raise RuntimeError(f"AR-EOF Branch{branch} 无可用配置")
    return best

def recursive_ar_predict(best,fd,split,branch):
    rows=[]
    for issue in fd[split]["issues"]:
        hist=observed_pc_history(fd,branch,pd.Timestamp(issue),best["lags"])
        if hist is None:
            raise RuntimeError(
                f"AR history缺失: branch={branch}, issue={issue}"
            )
        rows.append(
            recursive_ar_pc_forecast(
                best["model"],hist,best["lags"],max(cfg.LEAD_DAYS)
            )
        )
    return np.stack(rows).astype(np.float32)


In [ ]:

# ===== 8. PyTorch 模型：参数匹配LSTM + 原4套Hybrid =====
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset,DataLoader

class PerStepConvEncoder(nn.Module):
    def __init__(self,feature_dim,variant,branch,dropout):
        super().__init__()
        self.variant=variant
        self.feature_dim=int(feature_dim)
        self.relu=nn.ReLU()
        self.drop=nn.Dropout(dropout)

        if variant=="conv":
            c1=16 if branch==1 else 32
            c2=32 if branch==1 else 64
            self.conv1=nn.Conv1d(1,c1,3,padding=1)
            self.conv2=nn.Conv1d(c1,c2,3,padding=1)
            self.out_dim=c2*self.feature_dim

        elif variant=="tcn":
            c=16 if branch==1 else 32
            k=3 if branch==1 else 5
            hidden=32 if branch==1 else 64
            self.k=k
            self.c=c

            self.conv1=nn.Conv1d(
                1,c,kernel_size=k,dilation=2,padding=0
            )
            self.bn1=nn.BatchNorm1d(c)

            effective=(k-1)*2+1
            conv_len=(self.feature_dim+2)-effective+1
            if conv_len<=0:
                raise RuntimeError("invalid TCN conv length")

            self.fc=nn.Linear(c*conv_len,hidden)
            self.out_dim=hidden
        else:
            raise ValueError(variant)

    def forward(self,x):
        B,T,F=x.shape
        if F!=self.feature_dim:
            raise RuntimeError(
                f"feature dim mismatch {F}!={self.feature_dim}"
            )

        z=x.reshape(B*T,1,F)

        if self.variant=="conv":
            z=self.drop(self.relu(self.conv1(z)))
            z=self.drop(self.relu(self.conv2(z)))
            z=z.flatten(1)
        else:
            z=torch.nn.functional.pad(z,(2,0))
            z=self.conv1(z)
            z=self.drop(self.relu(self.bn1(z)))
            z=z.flatten(1)

            if z.shape[1]!=self.fc.in_features:
                raise RuntimeError(
                    f"TCN flatten {z.shape[1]} != fc {self.fc.in_features}"
                )
            z=self.drop(self.relu(self.fc(z)))

        return z.reshape(B,T,-1)

class HybridNet(nn.Module):
    def __init__(self,feature_dim,n_modes,branch,conv_kind,rnn_kind):
        super().__init__()
        self.enc=PerStepConvEncoder(
            feature_dim,conv_kind,branch,cfg.DROPOUT
        )

        hidden=(
            (16 if branch==1 else 64)
            if conv_kind=="conv"
            else (32 if branch==1 else 64)
        )

        RNN=nn.LSTM if rnn_kind=="lstm" else nn.GRU
        self.rnn=RNN(
            self.enc.out_dim,
            hidden,
            batch_first=True
        )
        self.head=nn.Linear(
            hidden,
            len(cfg.LEAD_DAYS)*n_modes
        )
        self.n_modes=n_modes

    def forward(self,x):
        z=self.enc(x)
        o,_=self.rnn(z)
        return self.head(o[:,-1]).reshape(
            x.shape[0],len(cfg.LEAD_DAYS),self.n_modes
        )

class MatchedLSTM(nn.Module):
    def __init__(self,feature_dim,n_modes,hidden):
        super().__init__()
        self.hidden=int(hidden)
        self.rnn=nn.LSTM(
            feature_dim,self.hidden,batch_first=True
        )
        self.head=nn.Linear(
            self.hidden,
            len(cfg.LEAD_DAYS)*n_modes
        )
        self.n_modes=n_modes

    def forward(self,x):
        o,_=self.rnn(x)
        return self.head(o[:,-1]).reshape(
            x.shape[0],len(cfg.LEAD_DAYS),self.n_modes
        )

def nparams(model):
    return int(sum(p.numel() for p in model.parameters()))

def find_matched_lstm_hidden(feature_dim,n_modes,branch):
    target=HybridNet(
        feature_dim,n_modes,branch,
        "tcn","lstm"
    )
    target_n=nparams(target)

    best=None
    for h in range(8,513):
        m=MatchedLSTM(feature_dim,n_modes,h)
        n=nparams(m)
        rel=abs(n-target_n)/max(target_n,1)
        if best is None or rel<best["rel"]:
            best={
                "hidden":h,
                "lstm_params":n,
                "target_params":target_n,
                "rel":rel,
            }

    return best

MODEL_SPECS=[
    ("Matched-LSTM","matched","lstm"),
    ("EOF-Conv-LSTM","conv","lstm"),
    ("EOF-Conv-GRU","conv","gru"),
    ("EOF-TCN-LSTM","tcn","lstm"),
    ("EOF-TCN-GRU","tcn","gru"),
]

MATCH_INFO={}

def build_model(name,conv_kind,rnn_kind,branch,fd):
    feature_dim=fd["train"][f"X{branch}"].shape[-1]
    n_modes=fd[f"pp{branch}"].n_modes

    if conv_kind=="matched":
        key=(id(fd),branch)
        info=find_matched_lstm_hidden(
            feature_dim,n_modes,branch
        )
        MATCH_INFO[(branch,feature_dim,n_modes)]=info
        return MatchedLSTM(
            feature_dim,n_modes,info["hidden"]
        )

    return HybridNet(
        feature_dim,n_modes,branch,
        conv_kind,rnn_kind
    )

# ---------- architecture & parameter fairness preflight ----------
preflight=[]
param_fairness=[]

for fold,fd in FOLD_DATA.items():
    for name,conv_kind,rnn_kind in MODEL_SPECS:
        for branch in (1,2):
            m=build_model(name,conv_kind,rnn_kind,branch,fd)
            X=fd["train"][f"X{branch}"][:2]
            xb=torch.from_numpy(X).float()

            with torch.no_grad():
                y=m(xb)

            expected=(
                len(xb),
                len(cfg.LEAD_DAYS),
                fd[f"pp{branch}"].n_modes
            )

            if tuple(y.shape)!=expected:
                raise RuntimeError(
                    f"preflight fail fold={fold} model={name} branch={branch}: "
                    f"{tuple(y.shape)} != {expected}"
                )

            preflight.append({
                "折次":fold,
                "模型":name,
                "分支":branch,
                "输入形状":str(tuple(xb.shape)),
                "输出形状":str(tuple(y.shape)),
                "参数量":nparams(m),
            })

    # matched LSTM vs ETL parameter fairness
    for branch in (1,2):
        feature_dim=fd["train"][f"X{branch}"].shape[-1]
        n_modes=fd[f"pp{branch}"].n_modes

        etl=HybridNet(
            feature_dim,n_modes,branch,
            "tcn","lstm"
        )
        info=find_matched_lstm_hidden(
            feature_dim,n_modes,branch
        )

        param_fairness.append({
            "折次":fold,
            "分支":branch,
            "ETL参数量":info["target_params"],
            "匹配LSTM隐藏维度":info["hidden"],
            "匹配LSTM参数量":info["lstm_params"],
            "参数量相对差":info["rel"],
        })

preflight_df=pd.DataFrame(preflight)
preflight_df.to_csv(
    DATA_DIR/"04_神经网络结构预检.csv",
    index=False,encoding="utf-8-sig"
)

param_fairness_df=pd.DataFrame(param_fairness)
param_fairness_df.to_csv(
    DATA_DIR/"04_LSTM与ETL参数量公平性.csv",
    index=False,encoding="utf-8-sig"
)

if (param_fairness_df["参数量相对差"]>cfg.PARAM_MATCH_TOLERANCE).any():
    warnings.warn(
        "部分fold的Matched-LSTM与ETL参数量相差超过设定容差，"
        "请检查04_LSTM与ETL参数量公平性.csv。"
    )

print("[architecture preflight] ALL PASS")
display(param_fairness_df)


In [ ]:

# ===== 9. 训练 / checkpoint / resume =====
from contextlib import nullcontext

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def run_dir(fold,model,seed):
    p=CKPT_DIR/f"第{fold}折"/model/f"种子{seed}"
    p.mkdir(parents=True,exist_ok=True)
    return p

def safe_save_model(model,path,meta):
    torch.save({
        "state_dict":{
            k:v.detach().cpu()
            for k,v in model.state_dict().items()
        },
        "meta":meta
    },path)

def train_one(model,fd,branch,max_epochs,seed):
    seed_all(seed)
    model=model.to(device)

    Xtr=torch.from_numpy(fd["train"][f"X{branch}"]).float()
    Ytr=torch.from_numpy(fd["train"][f"Y{branch}_pc"]).float()
    Xva=torch.from_numpy(fd["val"][f"X{branch}"]).float()
    Yva=torch.from_numpy(fd["val"][f"Y{branch}_pc"]).float()

    train_loader=DataLoader(
        TensorDataset(Xtr,Ytr),
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    val_loader=DataLoader(
        TensorDataset(Xva,Yva),
        batch_size=cfg.BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )

    opt=torch.optim.AdamW(
        model.parameters(),
        lr=cfg.LR,
        weight_decay=cfg.WEIGHT_DECAY
    )
    sched=torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt,
        mode="min",
        factor=cfg.LR_FACTOR,
        patience=cfg.LR_PATIENCE,
        min_lr=1e-5
    )
    loss_fn=nn.MSELoss()

    amp_enabled=bool(
        cfg.USE_AMP and
        torch.cuda.is_available()
    )
    scaler=torch.amp.GradScaler(
        "cuda",
        enabled=amp_enabled
    )

    best_loss=np.inf
    best_state=None
    best_epoch=-1
    patience=0
    hist=[]

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    t0=time.perf_counter()

    for ep in range(1,max_epochs+1):
        model.train()
        tr_sum=0.0
        tr_n=0

        for xb,yb in train_loader:
            xb=xb.to(device,non_blocking=True)
            yb=yb.to(device,non_blocking=True)

            opt.zero_grad(set_to_none=True)

            ctx=torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=amp_enabled
            ) if torch.cuda.is_available() else nullcontext()

            with ctx:
                pred=model(xb)
                loss=loss_fn(pred,yb)

            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            tr_sum += loss.detach().item()*len(xb)
            tr_n += len(xb)

        train_loss=tr_sum/max(tr_n,1)

        model.eval()
        va_sum=0.0
        va_n=0

        with torch.no_grad():
            for xb,yb in val_loader:
                xb=xb.to(device)
                yb=yb.to(device)

                ctx=torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                    enabled=amp_enabled
                ) if torch.cuda.is_available() else nullcontext()

                with ctx:
                    pred=model(xb)
                    loss=loss_fn(pred,yb)

                va_sum += loss.detach().item()*len(xb)
                va_n += len(xb)

        val_loss=va_sum/max(va_n,1)
        sched.step(val_loss)

        hist.append({
            "epoch":ep,
            "train_loss":train_loss,
            "val_loss":val_loss,
            "lr":opt.param_groups[0]["lr"]
        })

        if val_loss < best_loss-1e-7:
            best_loss=val_loss
            best_epoch=ep
            best_state={
                k:v.detach().cpu().clone()
                for k,v in model.state_dict().items()
            }
            patience=0
        else:
            patience+=1

        if ep==1 or ep%10==0:
            print(
                f"seed={seed} B{branch} ep={ep:03d} "
                f"train={train_loss:.6f} val={val_loss:.6f}"
            )

        if patience>=cfg.EARLY_STOP_PATIENCE:
            break

    elapsed=time.perf_counter()-t0

    if best_state is None:
        raise RuntimeError("no best checkpoint")

    model.load_state_dict(best_state)
    model=model.to(device).eval()

    peak_gb=(
        torch.cuda.max_memory_allocated()/1024**3
        if torch.cuda.is_available()
        else np.nan
    )

    return model,pd.DataFrame(hist),{
        "best_epoch":best_epoch,
        "best_val_loss":best_loss,
        "train_seconds":elapsed,
        "peak_gpu_gb":peak_gb,
    }

def predict_neural(model,fd,split,branch):
    X=torch.from_numpy(
        fd[split][f"X{branch}"]
    ).float()

    loader=DataLoader(
        TensorDataset(X),
        batch_size=cfg.BATCH_SIZE,
        shuffle=False
    )

    out=[]
    model.eval()
    with torch.no_grad():
        for (xb,) in loader:
            xb=xb.to(device)
            p=model(xb).float().cpu().numpy()
            out.append(p)

    return np.concatenate(out,axis=0)


In [ ]:

# ===== 10. 正式 blocked-CV + baseline 实验 =====
all_sample_metrics=[]
run_summary=[]
training_history=[]

def save_run_metrics(df):
    partial=pd.concat(all_sample_metrics,ignore_index=True) if all_sample_metrics else pd.DataFrame()
    if len(partial):
        partial.to_csv(
            DATA_DIR/"05_逐样本预测误差_中间保存.csv",
            index=False,encoding="utf-8-sig"
        )

for f in FOLDS:
    fold=f["fold"]
    fd=FOLD_DATA[fold]

    print("\n"+"#"*100)
    print("FOLD",fold)

    # ---------- Persistence ----------
    p1,p2=persistence_predict(fd["test"])
    met=evaluate_predictions(
        fold,"Persistence",-1,
        p1,p2,
        fd["test"]["Y1_raw"],
        fd["test"]["Y2_raw"],
        fd["test"]["issues"],
        fd["test"]["target_dates"]
    )
    all_sample_metrics.append(met)

    # ---------- Climatology ----------
    c1=fit_climatology(fd,1)
    c2=fit_climatology(fd,2)
    p1=climatology_predict(c1,fd["test"]["target_dates"])
    p2=climatology_predict(c2,fd["test"]["target_dates"])

    met=evaluate_predictions(
        fold,"Climatology",-1,
        p1,p2,
        fd["test"]["Y1_raw"],
        fd["test"]["Y2_raw"],
        fd["test"]["issues"],
        fd["test"]["target_dates"]
    )
    all_sample_metrics.append(met)

    # ---------- Recursive AR-EOF ----------
    ar1=fit_recursive_ar_branch(fd,f,1)
    ar2=fit_recursive_ar_branch(fd,f,2)

    pc1=recursive_ar_predict(ar1,fd,"test",1)
    pc2=recursive_ar_predict(ar2,fd,"test",2)
    p1=fd["pp1"].pc_to_path(pc1)
    p2=fd["pp2"].pc_to_path(pc2)

    met=evaluate_predictions(
        fold,"AR-EOF",-1,
        p1,p2,
        fd["test"]["Y1_raw"],
        fd["test"]["Y2_raw"],
        fd["test"]["issues"],
        fd["test"]["target_dates"]
    )
    all_sample_metrics.append(met)

    run_summary.append({
        "折次":fold,
        "模型":"AR-EOF",
        "随机种子":-1,
        "分支1_AR_alpha":ar1["alpha"],
        "分支2_AR_alpha":ar2["alpha"],
        "分支1_AR_lag":ar1["lags"],
        "分支2_AR_lag":ar2["lags"],
        "分支1最佳epoch":np.nan,
        "分支2最佳epoch":np.nan,
        "训练秒数":0.0,
        "峰值GPU显存_GB":0.0,
        "状态":"完成"
    })

    save_run_metrics(met)

    # ---------- Neural ----------
    for name,conv_kind,rnn_kind in MODEL_SPECS:
        for seed in cfg.SEEDS:
            print("\n"+"="*88)
            print(f"fold={fold} {name} seed={seed}")

            rd=run_dir(fold,name,seed)
            result_json=rd/"运行完成.json"
            pred_npz=rd/"测试集预测结果.npz"

            if (
                (not cfg.FORCE_RETRAIN) and
                result_json.exists() and
                pred_npz.exists()
            ):
                print("[resume] 已完成，直接读取预测结果")
                info=json.loads(result_json.read_text(encoding="utf-8"))
                with np.load(pred_npz) as z:
                    p1=z["pred1"]
                    p2=z["pred2"]

                met=evaluate_predictions(
                    fold,name,seed,
                    p1,p2,
                    fd["test"]["Y1_raw"],
                    fd["test"]["Y2_raw"],
                    fd["test"]["issues"],
                    fd["test"]["target_dates"]
                )
                all_sample_metrics.append(met)
                run_summary.append(info)
                save_run_metrics(met)
                continue

            net1=build_model(name,conv_kind,rnn_kind,1,fd)
            net2=build_model(name,conv_kind,rnn_kind,2,fd)

            net1,h1,b1=train_one(
                net1,fd,1,cfg.MAX_EPOCHS_B1,seed
            )
            net2,h2,b2=train_one(
                net2,fd,2,cfg.MAX_EPOCHS_B2,seed
            )

            pc1=predict_neural(net1,fd,"test",1)
            pc2=predict_neural(net2,fd,"test",2)

            p1=fd["pp1"].pc_to_path(pc1)
            p2=fd["pp2"].pc_to_path(pc2)

            np.savez_compressed(
                pred_npz,
                pred1=p1.astype(np.float32),
                pred2=p2.astype(np.float32),
                issues=fd["test"]["issues"].astype(str),
                target_dates=fd["test"]["target_dates"].astype(str),
            )

            safe_save_model(
                net1,rd/"分支1_最佳权重.pt",
                {
                    "fold":fold,"model":name,"seed":seed,"branch":1,
                    "n_modes":fd["pp1"].n_modes
                }
            )
            safe_save_model(
                net2,rd/"分支2_最佳权重.pt",
                {
                    "fold":fold,"model":name,"seed":seed,"branch":2,
                    "n_modes":fd["pp2"].n_modes
                }
            )

            h1["折次"]=fold
            h1["模型"]=name
            h1["随机种子"]=seed
            h1["分支"]=1

            h2["折次"]=fold
            h2["模型"]=name
            h2["随机种子"]=seed
            h2["分支"]=2

            training_history += [
                h1.rename(columns={
                    "epoch":"epoch","train_loss":"train_loss",
                    "val_loss":"val_loss","lr":"lr"
                }),
                h2.rename(columns={
                    "epoch":"epoch","train_loss":"train_loss",
                    "val_loss":"val_loss","lr":"lr"
                })
            ]

            info={
                "折次":fold,
                "模型":name,
                "随机种子":seed,
                "分支1_AR_alpha":None,
                "分支2_AR_alpha":None,
                "分支1_AR_lag":None,
                "分支2_AR_lag":None,
                "分支1最佳epoch":int(b1["best_epoch"]),
                "分支2最佳epoch":int(b2["best_epoch"]),
                "训练秒数":float(b1["train_seconds"]+b2["train_seconds"]),
                "峰值GPU显存_GB":float(max(b1["peak_gpu_gb"],b2["peak_gpu_gb"])),
                "状态":"完成"
            }

            result_json.write_text(
                json.dumps(info,indent=2,ensure_ascii=False),
                encoding="utf-8"
            )

            run_summary.append(info)

            met=evaluate_predictions(
                fold,name,seed,
                p1,p2,
                fd["test"]["Y1_raw"],
                fd["test"]["Y2_raw"],
                fd["test"]["issues"],
                fd["test"]["target_dates"]
            )
            all_sample_metrics.append(met)

            save_run_metrics(met)

# save final raw experiment tables
sample_metrics=pd.concat(all_sample_metrics,ignore_index=True)
sample_metrics.to_csv(
    DATA_DIR/"05_逐样本预测误差.csv",
    index=False,encoding="utf-8-sig"
)

run_summary_df=pd.DataFrame(run_summary)
run_summary_df.to_csv(
    DATA_DIR/"06_训练运行记录.csv",
    index=False,encoding="utf-8-sig"
)

if training_history:
    history_df=pd.concat(training_history,ignore_index=True)
    history_df.to_csv(
        DATA_DIR/"07_神经网络训练历史.csv",
        index=False,encoding="utf-8-sig"
    )

print("blocked CV complete")


In [ ]:

# ===== 11. 汇总 Reviewer 实验结果 =====
# 每个 model / fold / seed / lead 的样本平均
metric_cols=[
    "横向误差平均值_km",
    "横向误差中位数_km",
    "横向误差RMSE_km",
    "横向误差P90_km",
    "横向误差P95_km",
    "分支1经度RMSE_度",
    "分支1经度MAE_度",
    "分支2纬度RMSE_度",
    "分支2纬度MAE_度",
]

per_run=(
    sample_metrics
    .groupby(["折次","模型","随机种子","提前期_天"],as_index=False)[metric_cols]
    .mean()
)

per_run.to_csv(
    DATA_DIR/"08_各折各随机种子各提前期指标.csv",
    index=False,encoding="utf-8-sig"
)

# 总体均值：先fold/seed run，再总体
summary=(
    per_run
    .groupby(["模型","提前期_天"],as_index=False)
    .agg(
        横向误差平均值_km=("横向误差平均值_km","mean"),
        横向误差RMSE_km=("横向误差RMSE_km","mean"),
        横向误差中位数_km=("横向误差中位数_km","mean"),
        横向误差P90_km=("横向误差P90_km","mean"),
        横向误差P95_km=("横向误差P95_km","mean"),
        横向误差RMSE标准差_km=("横向误差RMSE_km","std"),
        运行数=("横向误差RMSE_km","size"),
    )
)

# persistence skill
pers=summary.loc[
    summary["模型"]=="Persistence",
    ["提前期_天","横向误差RMSE_km"]
].rename(columns={
    "横向误差RMSE_km":"Persistence_RMSE_km"
})

summary=summary.merge(pers,on="提前期_天",how="left")
summary["相对Persistence技能评分"]=(
    1.0-summary["横向误差RMSE_km"]/summary["Persistence_RMSE_km"]
)

summary.to_csv(
    DATA_DIR/"09_各模型总体性能汇总.csv",
    index=False,encoding="utf-8-sig"
)

baseline_table=summary[
    summary["模型"].isin([
        "Persistence","Climatology","AR-EOF","Simple-LSTM"
    ])
].copy()

baseline_table.to_csv(
    DATA_DIR/"10_审稿人要求的基线模型比较.csv",
    index=False,encoding="utf-8-sig"
)

skill_table=summary[
    summary["模型"]!="Persistence"
][[
    "模型","提前期_天","横向误差RMSE_km",
    "Persistence_RMSE_km","相对Persistence技能评分"
]]

skill_table.to_csv(
    DATA_DIR/"11_相对持续性预测技能评分.csv",
    index=False,encoding="utf-8-sig"
)

# fold-wise 90d table
fold90=(
    per_run[per_run["提前期_天"]==90]
    .groupby(["折次","模型"],as_index=False)
    ["横向误差RMSE_km"].mean()
)
fold90.to_csv(
    DATA_DIR/"12_各折90天泛化误差.csv",
    index=False,encoding="utf-8-sig"
)

display(summary.sort_values(["提前期_天","横向误差RMSE_km"]))


# R3: 主模型与Reviewer四基线专用表
reviewer_models=[
    "Persistence",
    "Climatology",
    "AR-EOF",
    "Matched-LSTM",
    cfg.PRIMARY_MODEL,
]

reviewer_comparison=summary[
    summary["模型"].isin(reviewer_models)
].copy()

reviewer_comparison.to_csv(
    DATA_DIR/"10_主模型与审稿人四类基线比较.csv",
    index=False,encoding="utf-8-sig"
)

# 5个神经结构专用表（Matched-LSTM + 4 hybrids）
neural_models=[
    "Matched-LSTM",
    "EOF-Conv-LSTM",
    "EOF-Conv-GRU",
    "EOF-TCN-LSTM",
    "EOF-TCN-GRU",
]

neural_structure_summary=summary[
    summary["模型"].isin(neural_models)
].copy()

neural_structure_summary.to_csv(
    DATA_DIR/"12_不同神经网络结构性能汇总.csv",
    index=False,encoding="utf-8-sig"
)


In [ ]:

# ===== 12. R3-1 预测误差不确定性 + 网络结构差异 =====
from itertools import combinations

rng_boot=np.random.default_rng(cfg.BOOTSTRAP_SEED)

def moving_block_indices(n,block_len,rng):
    if n<=0:
        return np.array([],dtype=int)
    block_len=max(1,min(int(block_len),n))
    out=[]
    while len(out)<n:
        s=int(rng.integers(0,max(1,n-block_len+1)))
        out.extend(range(s,min(n,s+block_len)))
    return np.asarray(out[:n],dtype=int)

def hierarchical_model_ci(model,lead,n_boot=None):
    """
    神经模型：seed resampling + issue-date moving-block bootstrap
    确定性基线：只做issue-date moving-block bootstrap
    最后对fold等权平均。
    """
    n_boot=cfg.BOOTSTRAP_N if n_boot is None else n_boot
    q=sample_metrics[
        (sample_metrics["模型"]==model)&
        (sample_metrics["提前期_天"]==lead)
    ].copy()

    if len(q)==0:
        return None

    point_fold=[]
    for fold in sorted(q["折次"].unique()):
        qq=q[q["折次"]==fold]
        seed_issue=(
            qq.groupby(["随机种子","起报日期"],as_index=False)
            ["横向误差RMSE_km"].mean()
        )
        issue_avg=(
            seed_issue.groupby("起报日期")
            ["横向误差RMSE_km"].mean()
        )
        point_fold.append(float(issue_avg.mean()))

    point=float(np.mean(point_fold))

    reps=[]
    for _ in range(n_boot):
        fold_vals=[]
        for fold in sorted(q["折次"].unique()):
            qq=q[q["折次"]==fold].copy()
            seeds=sorted(qq["随机种子"].unique())

            # deterministic baseline: seed=-1
            if seeds==[-1]:
                selected=qq.groupby("起报日期",as_index=False)["横向误差RMSE_km"].mean()
            else:
                boot_seeds=rng_boot.choice(seeds,size=len(seeds),replace=True)
                chunks=[]
                for s in boot_seeds:
                    chunks.append(
                        qq[qq["随机种子"]==s][["起报日期","横向误差RMSE_km"]]
                    )
                selected=(
                    pd.concat(chunks,ignore_index=True)
                    .groupby("起报日期",as_index=False)
                    ["横向误差RMSE_km"].mean()
                )

            selected=selected.sort_values("起报日期")
            vals=selected["横向误差RMSE_km"].to_numpy(float)

            ids=moving_block_indices(
                len(vals),
                cfg.PRED_BOOT_BLOCK_DAYS,
                rng_boot
            )
            fold_vals.append(float(np.mean(vals[ids])))

        reps.append(float(np.mean(fold_vals)))

    lo,hi=np.percentile(reps,[2.5,97.5])

    return {
        "模型":model,
        "提前期_天":lead,
        "横向误差RMSE均值_km":point,
        "95CI下限_km":float(lo),
        "95CI上限_km":float(hi),
        "Bootstrap次数":n_boot,
        "时间块长度_天":cfg.PRED_BOOT_BLOCK_DAYS,
    }

unc_rows=[]
all_models=[
    "Persistence","Climatology","AR-EOF",
    "Matched-LSTM",
    "EOF-Conv-LSTM","EOF-Conv-GRU",
    "EOF-TCN-LSTM","EOF-TCN-GRU",
]

for m in all_models:
    for lead in cfg.LEAD_DAYS:
        r=hierarchical_model_ci(m,lead)
        if r:
            unc_rows.append(r)

prediction_uncertainty=pd.DataFrame(unc_rows)
prediction_uncertainty.to_csv(
    DATA_DIR/"13_预测误差Hierarchical_BlockBootstrap_95CI.csv",
    index=False,encoding="utf-8-sig"
)

# ---------- 不同网络结构 pairwise uncertainty ----------
neural_models=[
    "Matched-LSTM",
    "EOF-Conv-LSTM","EOF-Conv-GRU",
    "EOF-TCN-LSTM","EOF-TCN-GRU"
]

# 先seed-average，避免把seed当成独立时间样本
neural_issue=(
    sample_metrics[
        sample_metrics["模型"].isin(neural_models)
    ]
    .groupby(
        ["折次","模型","起报日期","提前期_天"],
        as_index=False
    )["横向误差RMSE_km"].mean()
)

def paired_architecture_bootstrap(model_a,model_b,lead,n_boot=None):
    n_boot=cfg.BOOTSTRAP_N if n_boot is None else n_boot
    qa=neural_issue[
        (neural_issue["模型"]==model_a)&
        (neural_issue["提前期_天"]==lead)
    ]
    qb=neural_issue[
        (neural_issue["模型"]==model_b)&
        (neural_issue["提前期_天"]==lead)
    ]

    fold_diffs=[]
    fold_frames=[]

    for fold in sorted(set(qa["折次"]) & set(qb["折次"])):
        a=qa[qa["折次"]==fold][["起报日期","横向误差RMSE_km"]].rename(
            columns={"横向误差RMSE_km":"A"}
        )
        b=qb[qb["折次"]==fold][["起报日期","横向误差RMSE_km"]].rename(
            columns={"横向误差RMSE_km":"B"}
        )
        z=a.merge(b,on="起报日期").sort_values("起报日期")
        if len(z)==0:
            continue
        z["diff"]=z["A"]-z["B"]  # A-B; <0 means A better
        fold_frames.append((fold,z))
        fold_diffs.append(float(z["diff"].mean()))

    if not fold_frames:
        return None

    point=float(np.mean(fold_diffs))
    reps=[]

    for _ in range(n_boot):
        vals=[]
        for fold,z in fold_frames:
            d=z["diff"].to_numpy(float)
            ids=moving_block_indices(
                len(d),cfg.PRED_BOOT_BLOCK_DAYS,rng_boot
            )
            vals.append(float(np.mean(d[ids])))
        reps.append(float(np.mean(vals)))

    lo,hi=np.percentile(reps,[2.5,97.5])
    p_a_better=float(np.mean(np.asarray(reps)<0))

    return {
        "模型A":model_a,
        "模型B":model_b,
        "提前期_天":lead,
        "A减B误差差_km":point,
        "95CI下限_km":float(lo),
        "95CI上限_km":float(hi),
        "P_A优于B":p_a_better,
        "CI是否跨0":bool(lo<=0<=hi),
    }

pair_rows=[]
for a,b in combinations(neural_models,2):
    for lead in cfg.LEAD_DAYS:
        r=paired_architecture_bootstrap(a,b,lead)
        if r:
            pair_rows.append(r)

network_difference_uncertainty=pd.DataFrame(pair_rows)
network_difference_uncertainty.to_csv(
    DATA_DIR/"14_不同网络结构差异_PairedBlockBootstrap_95CI.csv",
    index=False,encoding="utf-8-sig"
)

# 主图使用：相对ETL差异
vs_etl=[]
for m in neural_models:
    if m==cfg.PRIMARY_MODEL:
        continue
    for lead in cfg.LEAD_DAYS:
        r=paired_architecture_bootstrap(m,cfg.PRIMARY_MODEL,lead)
        if r:
            vs_etl.append(r)

network_vs_etl=pd.DataFrame(vs_etl)
network_vs_etl.to_csv(
    DATA_DIR/"14_网络结构相对ETL差异_95CI.csv",
    index=False,encoding="utf-8-sig"
)

display(prediction_uncertainty)
display(network_vs_etl)


In [ ]:

# ===== 13. R3-2 黑潮轴线识别 / 定位不确定性 =====
from scipy.interpolate import RegularGridInterpolator

EARTH_R_KM=6371.0088

def wrap_pi2(a):
    return (a+np.pi)%(2*np.pi)-np.pi

def angle_east_rad2(lon1,lat1,lon2,lat2):
    p1,p2=np.deg2rad(lat1),np.deg2rad(lat2)
    dl=np.deg2rad(lon2-lon1)
    y=np.sin(dl)*np.cos(p2)
    x=np.cos(p1)*np.sin(p2)-np.sin(p1)*np.cos(p2)*np.cos(dl)
    nav=np.arctan2(y,x)
    return wrap_pi2(np.pi/2-nav)

def forward_point2(lon,lat,east_angle,distance_km):
    nav=np.pi/2-east_angle
    phi1,lam1=np.deg2rad(lat),np.deg2rad(lon)
    d=distance_km/EARTH_R_KM
    phi2=np.arcsin(
        np.sin(phi1)*np.cos(d)+
        np.cos(phi1)*np.sin(d)*np.cos(nav)
    )
    lam2=lam1+np.arctan2(
        np.sin(nav)*np.sin(d)*np.cos(phi1),
        np.cos(d)-np.sin(phi1)*np.sin(phi2)
    )
    return float(np.rad2deg(lam2)),float(np.rad2deg(phi2))

SECTIONS={
    "A":((121.50,23.50),(125.00,22.75)),
    "B":((121.50,26.25),(124.25,24.25)),
    "C":((130.25,31.25),(132.50,27.00)),
    "D":((137.00,34.75),(138.00,27.75)),
    "E":((141.75,40.00),(145.00,31.50)),
}

def load_track_npz(path):
    with np.load(path,allow_pickle=False) as z:
        return np.asarray(z["lon"],float),np.asarray(z["lat"],float)

def point_segment_distance_km(lon,lat,a,b):
    # local projection around segment midpoint
    lon0=(a[0]+b[0])/2
    lat0=(a[1]+b[1])/2
    c=np.cos(np.deg2rad(lat0))
    P=np.array([(lon-lon0)*111.32*c,(lat-lat0)*111.32])
    A=np.array([(a[0]-lon0)*111.32*c,(a[1]-lat0)*111.32])
    B=np.array([(b[0]-lon0)*111.32*c,(b[1]-lat0)*111.32])
    AB=B-A
    den=float(np.dot(AB,AB))
    if den<=1e-12:
        return float(np.linalg.norm(P-A))
    t=float(np.clip(np.dot(P-A,AB)/den,0,1))
    Q=A+t*AB
    return float(np.linalg.norm(P-Q))

def local_axis_profile(field_lon,field_lat,speed_itp,lon,lat,tangent):
    offsets=np.linspace(
        -cfg.AXIS_PROFILE_HALF_KM,
        cfg.AXIS_PROFILE_HALF_KM,
        cfg.AXIS_PROFILE_N
    )
    xs=[];ys=[]
    for off in offsets:
        if abs(off)<1e-12:
            x,y=lon,lat
        else:
            b=tangent+(np.pi/2 if off>0 else -np.pi/2)
            x,y=forward_point2(lon,lat,b,abs(float(off)))
        xs.append(x);ys.append(y)

    pts=np.column_stack([ys,xs])
    sp=speed_itp(pts)
    valid=np.isfinite(sp)

    if not valid.any():
        return None

    sp2=np.asarray(sp,float)
    sp2[~valid]=np.nan
    k=int(np.nanargmax(sp2))
    peak=float(sp2[k])

    if peak<=0 or not np.isfinite(peak):
        return None

    threshold=cfg.AXIS_CORE_FRAC*peak

    # 只取包含主峰的连续 near-core 区间
    left=k
    right=k
    while left-1>=0 and np.isfinite(sp2[left-1]) and sp2[left-1]>=threshold:
        left-=1
    while right+1<len(sp2) and np.isfinite(sp2[right+1]) and sp2[right+1]>=threshold:
        right+=1

    width=float(offsets[right]-offsets[left])

    center_speed=float(
        speed_itp([[lat,lon]])[0]
    )
    ridge_ratio=float(center_speed/peak) if np.isfinite(center_speed) else np.nan

    # secondary peak ambiguity
    secondary=np.nan
    for j,v in enumerate(sp2):
        if not np.isfinite(v):
            continue
        if abs(offsets[j]-offsets[k]) < cfg.AXIS_SECONDARY_PEAK_SEP_KM:
            continue
        if j==0 or j==len(sp2)-1:
            continue
        if v>=sp2[j-1] and v>=sp2[j+1]:
            secondary=v if not np.isfinite(secondary) else max(secondary,v)

    secondary_ratio=float(secondary/peak) if np.isfinite(secondary) else 0.0
    ambiguous=secondary_ratio>=cfg.AXIS_AMBIGUOUS_RATIO

    return {
        "定位宽度_km":width,
        "中心速度脊比":ridge_ratio,
        "第二峰比":secondary_ratio,
        "多峰歧义":bool(ambiguous),
    }

axis_daily_csv=DATA_DIR/"15_黑潮轴线定位不确定性_逐日.csv"
axis_section_csv=DATA_DIR/"16_黑潮轴线定位不确定性_断面逐日.csv"

if axis_daily_csv.exists() and axis_section_csv.exists() and not cfg.FORCE_RETRAIN:
    axis_daily=pd.read_csv(axis_daily_csv,parse_dates=["日期"])
    axis_section_daily=pd.read_csv(axis_section_csv,parse_dates=["日期"])
    print("[resume] 已读取轴线不确定性结果")
else:
    day_rows=[]
    sec_rows=[]

    track_files=sorted(FINAL_TRACK_DIR.glob("*.npz"))

    for ii,tp in enumerate(track_files,1):
        dt=pd.Timestamp(tp.stem)
        cp=CACHE50_DIR/f"{dt:%Y%m%d}.npz"
        if not cp.exists():
            continue

        lonp,latp=load_track_npz(tp)

        with np.load(cp) as z:
            glon=np.asarray(z["lon"],float)
            glat=np.asarray(z["lat"],float)
            speed=np.hypot(
                np.asarray(z["u"],float),
                np.asarray(z["v"],float)
            )

        sitp=RegularGridInterpolator(
            (glat,glon),speed,
            bounds_error=False,fill_value=np.nan
        )

        point_records=[]

        for i in range(
            2,max(2,len(lonp)-2),
            max(1,cfg.AXIS_POINT_STRIDE)
        ):
            if i>=len(lonp)-2:
                break

            tangent=angle_east_rad2(
                lonp[i-1],latp[i-1],
                lonp[i+1],latp[i+1]
            )

            r=local_axis_profile(
                glon,glat,sitp,
                lonp[i],latp[i],tangent
            )
            if r is None:
                continue
            r["lon"]=float(lonp[i])
            r["lat"]=float(latp[i])
            point_records.append(r)

        if not point_records:
            continue

        pdf=pd.DataFrame(point_records)

        day_rows.append({
            "日期":dt,
            "有效采样点数":len(pdf),
            "定位宽度中位数_km":float(pdf["定位宽度_km"].median()),
            "定位宽度P90_km":float(pdf["定位宽度_km"].quantile(.90)),
            "中心速度脊比中位数":float(pdf["中心速度脊比"].median()),
            "多峰歧义比例":float(pdf["多峰歧义"].mean()),
        })

        # A-E附近的定位宽度
        for sec,(a,b) in SECTIONS.items():
            d=np.array([
                point_segment_distance_km(x,y,a,b)
                for x,y in zip(pdf["lon"],pdf["lat"])
            ])
            k=int(np.argmin(d))
            rr=pdf.iloc[k]
            sec_rows.append({
                "日期":dt,
                "断面":sec,
                "距断面最近距离_km":float(d[k]),
                "定位宽度_km":float(rr["定位宽度_km"]),
                "中心速度脊比":float(rr["中心速度脊比"]),
                "第二峰比":float(rr["第二峰比"]),
                "多峰歧义":bool(rr["多峰歧义"]),
            })

        if ii%25==0:
            pd.DataFrame(day_rows).to_csv(
                axis_daily_csv,index=False,encoding="utf-8-sig"
            )
            pd.DataFrame(sec_rows).to_csv(
                axis_section_csv,index=False,encoding="utf-8-sig"
            )
            print(f"[axis uncertainty] {ii}/{len(track_files)}")

    axis_daily=pd.DataFrame(day_rows)
    axis_section_daily=pd.DataFrame(sec_rows)

    axis_daily.to_csv(
        axis_daily_csv,index=False,encoding="utf-8-sig"
    )
    axis_section_daily.to_csv(
        axis_section_csv,index=False,encoding="utf-8-sig"
    )

def block_boot_ci_series(values,block_len,n_boot=None,stat=np.mean):
    values=np.asarray(values,float)
    values=values[np.isfinite(values)]
    n_boot=cfg.BOOTSTRAP_N if n_boot is None else n_boot
    if len(values)==0:
        return np.nan,np.nan,np.nan

    point=float(stat(values))
    reps=[]
    rng=np.random.default_rng(cfg.BOOTSTRAP_SEED+17)

    for _ in range(n_boot):
        ids=moving_block_indices(
            len(values),block_len,rng
        )
        reps.append(float(stat(values[ids])))

    lo,hi=np.percentile(reps,[2.5,97.5])
    return point,float(lo),float(hi)

sec_unc_rows=[]
for sec in "ABCDE":
    q=axis_section_daily[
        axis_section_daily["断面"]==sec
    ].sort_values("日期")

    point,lo,hi=block_boot_ci_series(
        q["定位宽度_km"],
        cfg.AXIS_BOOT_BLOCK_DAYS
    )

    ap,alo,ahi=block_boot_ci_series(
        q["多峰歧义"].astype(float),
        cfg.AXIS_BOOT_BLOCK_DAYS
    )

    sec_unc_rows.append({
        "断面":sec,
        "定位宽度均值_km":point,
        "定位宽度95CI下限_km":lo,
        "定位宽度95CI上限_km":hi,
        "多峰歧义比例":ap,
        "多峰歧义95CI下限":alo,
        "多峰歧义95CI上限":ahi,
        "天数":len(q),
    })

axis_section_uncertainty=pd.DataFrame(sec_unc_rows)
axis_section_uncertainty.to_csv(
    DATA_DIR/"16_黑潮轴线定位不确定性_断面95CI.csv",
    index=False,encoding="utf-8-sig"
)

display(axis_section_uncertainty)


In [ ]:

# ===== 14. R3-3 趋势分析与断面指标不确定性 =====
# 该部分针对原稿1993-2024长期监测结果。
# 若本地尚未准备长期月尺度统计CSV，脚本只生成模板并明确标记“未运行”，不会用2023-2025日数据冒充长期结果。

LONGTERM_REQUIRED_COLUMNS=[
    "date",
    "A_distance_km",
    "B_distance_km",
    "C_distance_km",
    "D_distance_km",
    "E_distance_km",
    "KE143_lat_deg",
    "KE144_lat_deg",
    "SOR_SST_mean_C",
]

def create_longterm_template():
    if LONGTERM_SERIES_CSV.exists():
        return
    pd.DataFrame(columns=LONGTERM_REQUIRED_COLUMNS).to_csv(
        LONGTERM_SERIES_CSV,
        index=False,
        encoding="utf-8-sig"
    )
    print(
        "已生成长期不确定性输入模板：",
        LONGTERM_SERIES_CSV,
        "\n请从原1993-2024 MATLAB监测流程导出月尺度序列填入。"
    )

create_longterm_template()

longterm_ready=False
longterm_status={
    "长期输入文件":str(LONGTERM_SERIES_CSV),
    "状态":"未运行",
    "原因":"长期CSV不存在或无有效数据",
}

if LONGTERM_SERIES_CSV.exists():
    try:
        longterm=pd.read_csv(
            LONGTERM_SERIES_CSV,
            parse_dates=["date"]
        )
        if (
            len(longterm)>=120 and
            all(c in longterm.columns for c in LONGTERM_REQUIRED_COLUMNS)
        ):
            longterm=longterm.sort_values("date")
            longterm_ready=True
            longterm_status={
                "长期输入文件":str(LONGTERM_SERIES_CSV),
                "状态":"READY",
                "样本数":len(longterm),
                "开始":str(longterm["date"].min().date()),
                "结束":str(longterm["date"].max().date()),
            }
    except Exception as e:
        longterm_status={
            "长期输入文件":str(LONGTERM_SERIES_CSV),
            "状态":"未运行",
            "原因":repr(e),
        }

pd.DataFrame([longterm_status]).to_csv(
    DATA_DIR/"17_长期趋势与断面不确定性输入状态.csv",
    index=False,encoding="utf-8-sig"
)

def moving_block_bootstrap_rows(df,block_len,rng):
    n=len(df)
    ids=moving_block_indices(n,block_len,rng)
    return df.iloc[ids].copy()

def ols_slope_per_decade(date,value):
    date=pd.to_datetime(date)
    y=np.asarray(value,float)
    good=np.isfinite(y)
    date=date[good]
    y=y[good]

    if len(y)<12:
        return np.nan

    x=(date-date.min()).days.to_numpy()/365.2425
    slope=np.polyfit(x,y,1)[0]
    return float(slope*10.0)

trend_uncertainty=pd.DataFrame()
section_metric_uncertainty=pd.DataFrame()

if longterm_ready:
    rng_long=np.random.default_rng(cfg.BOOTSTRAP_SEED+33)

    # ---------- A-E section metrics ----------
    sec_rows=[]
    for sec in "ABCDE":
        col=f"{sec}_distance_km"
        vals=longterm[col].to_numpy(float)
        mu=np.nanmean(vals)
        anomaly=vals-mu

        point_mean=float(np.nanmean(vals))
        point_std=float(np.nanstd(anomaly,ddof=1))
        point_p05=float(np.nanpercentile(anomaly,5))
        point_p95=float(np.nanpercentile(anomaly,95))

        boot=[]
        for _ in range(cfg.BOOTSTRAP_N):
            b=moving_block_bootstrap_rows(
                longterm[[col]],
                cfg.MONTHLY_BOOT_BLOCK_MONTHS,
                rng_long
            )[col].to_numpy(float)

            bm=np.nanmean(b)
            ba=b-bm
            boot.append([
                bm,
                np.nanstd(ba,ddof=1),
                np.nanpercentile(ba,5),
                np.nanpercentile(ba,95),
            ])

        boot=np.asarray(boot,float)
        cis=np.nanpercentile(boot,[2.5,97.5],axis=0)

        sec_rows.append({
            "断面":sec,
            "长期平均距离_km":point_mean,
            "长期平均距离95CI下限_km":cis[0,0],
            "长期平均距离95CI上限_km":cis[1,0],
            "位移标准差_km":point_std,
            "位移标准差95CI下限_km":cis[0,1],
            "位移标准差95CI上限_km":cis[1,1],
            "位移P05_km":point_p05,
            "位移P05_95CI下限_km":cis[0,2],
            "位移P05_95CI上限_km":cis[1,2],
            "位移P95_km":point_p95,
            "位移P95_95CI下限_km":cis[0,3],
            "位移P95_95CI上限_km":cis[1,3],
        })

    section_metric_uncertainty=pd.DataFrame(sec_rows)
    section_metric_uncertainty.to_csv(
        DATA_DIR/"18_A至E断面指标_BlockBootstrap_95CI.csv",
        index=False,encoding="utf-8-sig"
    )

    # ---------- 趋势 ----------
    trend_specs=[
        ("A_distance_km","A断面径向距离","km/decade"),
        ("B_distance_km","B断面径向距离","km/decade"),
        ("C_distance_km","C断面径向距离","km/decade"),
        ("D_distance_km","D断面径向距离","km/decade"),
        ("E_distance_km","E断面径向距离","km/decade"),
        ("KE143_lat_deg","143°E黑潮延伸体纬度","deg/decade"),
        ("KE144_lat_deg","144°E黑潮延伸体纬度","deg/decade"),
        ("SOR_SST_mean_C","三陆近海SST","degC/decade"),
    ]

    trend_rows=[]

    for col,label,unit in trend_specs:
        # 论文SST同时关心1993-2022和1993-2024；其他序列给全时段
        periods=[("全时段",longterm)]
        if col=="SOR_SST_mean_C":
            periods=[
                ("1993-2022",longterm[longterm["date"]<"2023-01-01"]),
                ("1993-2024",longterm[longterm["date"]<"2025-01-01"]),
            ]

        for period,q in periods:
            q=q[["date",col]].dropna().sort_values("date")
            point=ols_slope_per_decade(q["date"],q[col])

            reps=[]
            for _ in range(cfg.BOOTSTRAP_N):
                b=moving_block_bootstrap_rows(
                    q,
                    cfg.MONTHLY_BOOT_BLOCK_MONTHS,
                    rng_long
                ).sort_values("date")
                reps.append(
                    ols_slope_per_decade(b["date"],b[col])
                )

            lo,hi=np.nanpercentile(reps,[2.5,97.5])

            trend_rows.append({
                "变量":label,
                "列名":col,
                "时期":period,
                "趋势每十年":point,
                "95CI下限":float(lo),
                "95CI上限":float(hi),
                "单位":unit,
                "月样本数":len(q),
            })

    trend_uncertainty=pd.DataFrame(trend_rows)
    trend_uncertainty.to_csv(
        DATA_DIR/"19_长期趋势_BlockBootstrap_95CI.csv",
        index=False,encoding="utf-8-sig"
    )

    display(section_metric_uncertainty)
    display(trend_uncertainty)

else:
    print(
        "长期1993-2024监测序列尚未导入，因此趋势/长期断面CI暂不计算。\n"
        "已生成模板：",LONGTERM_SERIES_CSV,
        "\n这不会影响blocked CV、baseline、预测误差CI和轴线定位不确定性。"
    )


In [ ]:

# ===== 15. R3论文图：原稿风格，中英文双版本 =====
import matplotlib.pyplot as plt

ZH_NAMES={
    "Persistence":"持续性预测",
    "Climatology":"气候态",
    "AR-EOF":"自回归EOF",
    "Matched-LSTM":"参数量匹配LSTM",
    "EOF-Conv-LSTM":"EOF-Conv-LSTM",
    "EOF-Conv-GRU":"EOF-Conv-GRU",
    "EOF-TCN-LSTM":"EOF-TCN-LSTM",
    "EOF-TCN-GRU":"EOF-TCN-GRU",
}

# ---------- 图1 Strict blocked hindcast ----------
def fig_blocked(lang):
    fig,ax=plt.subplots(figsize=(9.0,4.6))
    y=np.arange(len(FOLDS))[::-1]

    for yy,f in zip(y,FOLDS):
        train_start=pd.Timestamp(anchor_index["最晚目标日期"].min())

        ax.plot(
            [train_start,f["train_target_end"]],
            [yy,yy],
            lw=9,
            color=STYLE_COLORS["blue"],
            solid_capstyle="butt",
            label=("训练标签可用期" if lang=="zh" else "Training-label period")
        )
        ax.plot(
            [f["val_issue_start"],f["val_issue_end"]],
            [yy,yy],
            lw=9,
            color=STYLE_COLORS["orange"],
            solid_capstyle="butt",
            label=("验证起报期" if lang=="zh" else "Validation issue block")
        )
        ax.plot(
            [f["test_issue_start"],f["test_issue_end"]],
            [yy,yy],
            lw=9,
            color=STYLE_COLORS["green"],
            solid_capstyle="butt",
            label=("测试起报期" if lang=="zh" else "Test issue block")
        )

    ax.set_yticks(y)
    ax.set_yticklabels(
        [f"第{f['fold']}折" for f in FOLDS]
        if lang=="zh"
        else [f"Fold {f['fold']}" for f in FOLDS]
    )
    ax.set_xlabel("时间" if lang=="zh" else "Time")
    ax.set_title(
        "严格因果分块后报验证"
        if lang=="zh"
        else "Strict causal blocked hindcast validation"
    )

    h,l=ax.get_legend_handles_labels()
    uniq={}
    for hh,ll in zip(h,l):
        uniq.setdefault(ll,hh)
    ax.legend(uniq.values(),uniq.keys(),loc="upper left",ncol=3)
    fig.autofmt_xdate()
    fig.tight_layout()
    return fig

save_dual_language(
    fig_blocked,
    "图1_严格因果分块后报验证_中文",
    "Figure1_Strict_Causal_Blocked_Hindcast_English"
)

# ---------- 图2 主模型 vs Reviewer四基线 ----------
def fig_baseline(lang):
    fig,ax=plt.subplots(figsize=(8.6,5.6))

    order=[
        "Persistence",
        "Climatology",
        "AR-EOF",
        "Matched-LSTM",
        cfg.PRIMARY_MODEL,
    ]

    colors=[
        STYLE_COLORS["gray"],
        STYLE_COLORS["purple_gray"],
        STYLE_COLORS["orange"],
        STYLE_COLORS["pale_blue"],
        STYLE_COLORS["blue"],
    ]

    for m,col in zip(order,colors):
        q=summary[summary["模型"]==m].sort_values("提前期_天")
        if len(q)==0:
            continue
        ax.plot(
            q["提前期_天"],
            q["横向误差RMSE_km"],
            "-o",
            color=col,
            markerfacecolor="white",
            markeredgecolor=col,
            label=(ZH_NAMES[m] if lang=="zh" else m)
        )

    ax.set_xticks(cfg.LEAD_DAYS)
    ax.set_xlabel("预测提前期（天）" if lang=="zh" else "Forecast lead (days)")
    ax.set_ylabel("横向路径误差 RMSE（km）" if lang=="zh" else "Cross-track path RMSE (km)")
    ax.set_title(
        "主模型与审稿人要求基线的预测误差"
        if lang=="zh"
        else "Forecast error of the proposed model and reviewer-requested baselines"
    )
    ax.legend()
    fig.tight_layout()
    return fig

save_dual_language(
    fig_baseline,
    "图2_主模型与四类审稿基线比较_中文",
    "Figure2_Proposed_Model_vs_Reviewer_Baselines_English"
)

# ---------- 图3 预测误差95%CI ----------
def fig_pred_unc(lang):
    fig,ax=plt.subplots(figsize=(8.8,5.8))

    order=[
        "Persistence",
        "Climatology",
        "AR-EOF",
        "Matched-LSTM",
        cfg.PRIMARY_MODEL,
    ]
    colors=[
        STYLE_COLORS["gray"],
        STYLE_COLORS["purple_gray"],
        STYLE_COLORS["orange"],
        STYLE_COLORS["pale_blue"],
        STYLE_COLORS["blue"],
    ]

    offsets=np.linspace(-1.2,1.2,len(order))

    for off,m,col in zip(offsets,order,colors):
        q=prediction_uncertainty[
            prediction_uncertainty["模型"]==m
        ].sort_values("提前期_天")
        if len(q)==0:
            continue

        x=q["提前期_天"].to_numpy(float)+off
        y=q["横向误差RMSE均值_km"].to_numpy(float)
        lo=y-q["95CI下限_km"].to_numpy(float)
        hi=q["95CI上限_km"].to_numpy(float)-y

        ax.errorbar(
            x,y,yerr=np.vstack([lo,hi]),
            fmt="o-",
            capsize=3,
            color=col,
            markerfacecolor="white",
            markeredgecolor=col,
            label=(ZH_NAMES[m] if lang=="zh" else m)
        )

    ax.set_xticks(cfg.LEAD_DAYS)
    ax.set_xlabel("预测提前期（天）" if lang=="zh" else "Forecast lead (days)")
    ax.set_ylabel(
        "横向路径误差 RMSE（km，95% CI）"
        if lang=="zh"
        else "Cross-track path RMSE (km, 95% CI)"
    )
    ax.set_title(
        "预测误差的不确定性"
        if lang=="zh"
        else "Uncertainty in forecast error"
    )
    ax.legend()
    fig.tight_layout()
    return fig

save_dual_language(
    fig_pred_unc,
    "图3_预测误差不确定性95CI_中文",
    "Figure3_Forecast_Error_Uncertainty_English"
)

# ---------- 图4 网络结构相对ETL差异 ----------
def fig_arch_diff(lang):
    fig,ax=plt.subplots(figsize=(8.8,5.8))

    q=network_vs_etl.copy()
    model_order=[
        "Matched-LSTM",
        "EOF-Conv-LSTM",
        "EOF-Conv-GRU",
        "EOF-TCN-GRU",
    ]

    xbase=np.arange(len(model_order))
    width=.20

    for j,lead in enumerate(cfg.LEAD_DAYS):
        vals=[]
        los=[]
        his=[]

        for m in model_order:
            qq=q[
                (q["模型A"]==m)&
                (q["提前期_天"]==lead)
            ]
            if len(qq)==0:
                vals.append(np.nan);los.append(np.nan);his.append(np.nan)
                continue

            r=qq.iloc[0]
            vals.append(float(r["A减B误差差_km"]))
            los.append(float(r["A减B误差差_km"]-r["95CI下限_km"]))
            his.append(float(r["95CI上限_km"]-r["A减B误差差_km"]))

        xx=xbase+(j-1)*width
        ax.errorbar(
            xx,vals,
            yerr=np.vstack([los,his]),
            fmt="o",
            capsize=3,
            label=(f"{lead}天" if lang=="zh" else f"{lead} days")
        )

    ax.axhline(0,color=STYLE_COLORS["dark"],lw=1.0)
    ax.set_xticks(xbase)
    ax.set_xticklabels(
        [ZH_NAMES[m] for m in model_order]
        if lang=="zh"
        else model_order,
        rotation=18,ha="right"
    )
    ax.set_ylabel(
        "相对EOF-TCN-LSTM的RMSE差（km）"
        if lang=="zh"
        else "RMSE difference vs EOF-TCN-LSTM (km)"
    )
    ax.set_title(
        "不同网络结构之间的误差差异及95%置信区间"
        if lang=="zh"
        else "Error differences among neural architectures with 95% CIs"
    )
    ax.legend()
    fig.tight_layout()
    return fig

save_dual_language(
    fig_arch_diff,
    "图4_不同网络结构差异95CI_中文",
    "Figure4_Network_Architecture_Differences_English"
)

# ---------- 图5 黑潮轴定位不确定性 ----------
def fig_axis_unc(lang):
    fig,ax=plt.subplots(figsize=(7.6,5.3))
    q=axis_section_uncertainty.copy()
    x=np.arange(len(q))
    y=q["定位宽度均值_km"].to_numpy(float)
    lo=y-q["定位宽度95CI下限_km"].to_numpy(float)
    hi=q["定位宽度95CI上限_km"].to_numpy(float)-y

    ax.errorbar(
        x,y,
        yerr=np.vstack([lo,hi]),
        fmt="o-",
        capsize=3,
        color=STYLE_COLORS["blue"],
        markerfacecolor="white",
        markeredgecolor=STYLE_COLORS["blue"]
    )

    ax.set_xticks(x)
    ax.set_xticklabels(q["断面"])
    ax.set_xlabel("诊断断面" if lang=="zh" else "Diagnostic section")
    ax.set_ylabel(
        "局地轴定位宽度（km，95% CI）"
        if lang=="zh"
        else "Local axis localization width (km, 95% CI)"
    )
    ax.set_title(
        "黑潮轴线识别的局地定位不确定性"
        if lang=="zh"
        else "Local localization uncertainty of the detected Kuroshio axis"
    )
    fig.tight_layout()
    return fig

save_dual_language(
    fig_axis_unc,
    "图5_黑潮轴线定位不确定性_中文",
    "Figure5_Kuroshio_Axis_Localization_Uncertainty_English"
)

# ---------- 图6/7 长期断面与趋势（有输入才画） ----------
if longterm_ready and len(section_metric_uncertainty):
    def fig_section_unc(lang):
        fig,ax=plt.subplots(figsize=(7.6,5.3))
        q=section_metric_uncertainty.copy()
        x=np.arange(len(q))
        y=q["长期平均距离_km"].to_numpy(float)
        lo=y-q["长期平均距离95CI下限_km"].to_numpy(float)
        hi=q["长期平均距离95CI上限_km"].to_numpy(float)-y

        ax.errorbar(
            x,y,
            yerr=np.vstack([lo,hi]),
            fmt="o-",
            capsize=3,
            color=STYLE_COLORS["orange"],
            markerfacecolor="white",
            markeredgecolor=STYLE_COLORS["orange"]
        )
        ax.set_xticks(x)
        ax.set_xticklabels(q["断面"])
        ax.set_xlabel("诊断断面" if lang=="zh" else "Diagnostic section")
        ax.set_ylabel(
            "长期平均交点距离（km，95% CI）"
            if lang=="zh"
            else "Long-term mean intersection distance (km, 95% CI)"
        )
        ax.set_title(
            "A-E断面指标的不确定性"
            if lang=="zh"
            else "Uncertainty in section metrics A-E"
        )
        fig.tight_layout()
        return fig

    save_dual_language(
        fig_section_unc,
        "图6_A至E断面指标不确定性_中文",
        "Figure6_Uncertainty_in_Section_Metrics_English"
    )

if longterm_ready and len(trend_uncertainty):
    # 只画单位可统一的A-E趋势；SST/纬度保留在CSV避免双单位图
    trend_sec=trend_uncertainty[
        trend_uncertainty["列名"].str.match(r"^[ABCDE]_distance_km$")
    ].copy()

    if len(trend_sec):
        def fig_trend_unc(lang):
            fig,ax=plt.subplots(figsize=(7.8,5.3))
            x=np.arange(len(trend_sec))
            y=trend_sec["趋势每十年"].to_numpy(float)
            lo=y-trend_sec["95CI下限"].to_numpy(float)
            hi=trend_sec["95CI上限"].to_numpy(float)-y

            ax.errorbar(
                x,y,
                yerr=np.vstack([lo,hi]),
                fmt="o",
                capsize=3,
                color=STYLE_COLORS["blue"],
                markerfacecolor="white",
                markeredgecolor=STYLE_COLORS["blue"]
            )
            ax.axhline(0,color=STYLE_COLORS["dark"],lw=1.0)
            ax.set_xticks(x)
            ax.set_xticklabels(["A","B","C","D","E"][:len(x)])
            ax.set_xlabel("诊断断面" if lang=="zh" else "Diagnostic section")
            ax.set_ylabel(
                "径向距离趋势（km/10年，95% CI）"
                if lang=="zh"
                else "Radial-distance trend (km decade$^{-1}$, 95% CI)"
            )
            ax.set_title(
                "长期断面趋势及不确定性"
                if lang=="zh"
                else "Long-term section trends and uncertainty"
            )
            fig.tight_layout()
            return fig

        save_dual_language(
            fig_trend_unc,
            "图7_长期断面趋势不确定性_中文",
            "Figure7_LongTerm_Section_Trend_Uncertainty_English"
        )

print("中文图表:",FIG_ZH_DIR)
print("英文图表:",FIG_EN_DIR)


In [ ]:

# ===== 16. 自动生成实验结果摘要 =====
def dataframe_to_markdown_no_dependency(df,index=False):
    q=df.copy()
    if index:
        q=q.reset_index()

    def fmt(v):
        if pd.isna(v): return ""
        if isinstance(v,(float,np.floating)): return f"{float(v):.6g}"
        if isinstance(v,(pd.Timestamp,np.datetime64)): return str(pd.Timestamp(v))
        return str(v).replace("|",r"\|").replace("\n"," ")

    headers=[fmt(c) for c in q.columns]
    lines=[
        "| "+" | ".join(headers)+" |",
        "| "+" | ".join(["---"]*len(headers))+" |",
    ]
    for row in q.itertuples(index=False,name=None):
        lines.append("| "+" | ".join(fmt(v) for v in row)+" |")
    return "\n".join(lines)

primary=summary[
    summary["模型"]==cfg.PRIMARY_MODEL
].sort_values("提前期_天")

baseline_main=summary[
    summary["模型"].isin([
        "Persistence","Climatology","AR-EOF",
        "Matched-LSTM",cfg.PRIMARY_MODEL
    ])
].sort_values(["提前期_天","模型"])

result_lines=[
    "# 正式训练与审稿实验结果摘要",
    "",
    f"- 最终日尺度路径数：{len(dates)}",
    f"- 180-day有效anchors：{len(anchors)}",
    f"- 严格因果Blocked CV折数：{len(FOLDS)}",
    f"- 神经网络随机种子：{list(cfg.SEEDS)}",
    f"- 论文主模型：{cfg.PRIMARY_MODEL}",
    "",
    "## 严格时间泄漏审计",
    dataframe_to_markdown_no_dependency(leakage_df,index=False),
    "",
    "## 主模型与四类Reviewer baselines",
    dataframe_to_markdown_no_dependency(baseline_main,index=False),
    "",
    "## 预测误差95%CI",
    dataframe_to_markdown_no_dependency(
        prediction_uncertainty[
            prediction_uncertainty["模型"].isin([
                "Persistence","Climatology","AR-EOF",
                "Matched-LSTM",cfg.PRIMARY_MODEL
            ])
        ],
        index=False
    ),
    "",
    "## 不同网络结构相对ETL的误差差异",
    dataframe_to_markdown_no_dependency(network_vs_etl,index=False),
    "",
    "## 黑潮轴定位不确定性",
    dataframe_to_markdown_no_dependency(axis_section_uncertainty,index=False),
]

if longterm_ready:
    result_lines += [
        "",
        "## A-E断面指标不确定性",
        dataframe_to_markdown_no_dependency(section_metric_uncertainty,index=False),
        "",
        "## 长期趋势不确定性",
        dataframe_to_markdown_no_dependency(trend_uncertainty,index=False),
    ]
else:
    result_lines += [
        "",
        "## 长期趋势/断面不确定性",
        "尚未导入1993-2024月尺度监测序列；已生成长期输入模板。"
    ]

(REPORT_DIR/"R3_正式训练与审稿实验结果摘要_中文.md").write_text(
    "\n".join(result_lines),encoding="utf-8"
)

print("摘要 ->",REPORT_DIR/"R3_正式训练与审稿实验结果摘要_中文.md")


In [ ]:

# ===== 14. 最终 full-data frozen model retraining =====
if not cfg.TRAIN_FINAL_FROZEN_MODEL:
    print("TRAIN_FINAL_FROZEN_MODEL=False，跳过最终模型归档。")
else:
    # 使用所有anchors；尾部target-max时间块作为early-stopping validation
    tmax=np.array([np.datetime64(a["target_max"].date()) for a in anchors])
    unique=np.array(sorted(pd.unique(tmax)))

    val_start=pd.Timestamp(unique[-cfg.FINAL_VAL_DAYS]) if len(unique)>cfg.FINAL_VAL_DAYS else pd.Timestamp(unique[int(len(unique)*.85)])

    tr_idx=np.array([
        i for i,a in enumerate(anchors)
        if pd.Timestamp(a["target_max"])<val_start
    ],dtype=int)

    va_idx=np.array([
        i for i,a in enumerate(anchors)
        if pd.Timestamp(a["target_max"])>=val_start
    ],dtype=int)

    if len(tr_idx)<cfg.MIN_TRAIN_ANCHORS or len(va_idx)<10:
        raise RuntimeError(
            f"最终模型训练划分不足: train={len(tr_idx)} val={len(va_idx)}"
        )

    fake_fold={
        "fold":999,
        "train_idx":tr_idx,
        "val_idx":va_idx,
        "test_idx":va_idx,
    }
    fd=prepare_fold(fake_fold)

    spec={x[0]:x for x in MODEL_SPECS}[cfg.PRIMARY_MODEL]
    _,conv_kind,rnn_kind=spec

    final_rows=[]
    final_root=CKPT_DIR/"最终冻结模型"/cfg.PRIMARY_MODEL
    final_root.mkdir(parents=True,exist_ok=True)

    for seed in cfg.SEEDS:
        print("\n"+"="*80)
        print("FINAL",cfg.PRIMARY_MODEL,"seed",seed)

        sdir=final_root/f"种子{seed}"
        sdir.mkdir(parents=True,exist_ok=True)

        done=sdir/"训练完成.json"

        if done.exists() and not cfg.FORCE_RETRAIN:
            print("[resume] final frozen model already exists")
            final_rows.append(
                json.loads(done.read_text(encoding="utf-8"))
            )
            continue

        net1=build_model(cfg.PRIMARY_MODEL,conv_kind,rnn_kind,1,fd)
        net2=build_model(cfg.PRIMARY_MODEL,conv_kind,rnn_kind,2,fd)

        net1,h1,b1=train_one(net1,fd,1,cfg.MAX_EPOCHS_B1,seed)
        net2,h2,b2=train_one(net2,fd,2,cfg.MAX_EPOCHS_B2,seed)

        safe_save_model(
            net1,sdir/"分支1_最终权重.pt",
            {
                "model":cfg.PRIMARY_MODEL,
                "seed":seed,
                "branch":1,
                "validation_start":str(val_start.date()),
            }
        )
        safe_save_model(
            net2,sdir/"分支2_最终权重.pt",
            {
                "model":cfg.PRIMARY_MODEL,
                "seed":seed,
                "branch":2,
                "validation_start":str(val_start.date()),
            }
        )

        np.savez_compressed(
            sdir/"训练集预处理参数.npz",
            b1_mu=fd["pp1"].mu.astype(np.float32),
            b1_std=fd["pp1"].std.astype(np.float32),
            b1_V=fd["pp1"].V.astype(np.float32),
            b2_mu=fd["pp2"].mu.astype(np.float32),
            b2_std=fd["pp2"].std.astype(np.float32),
            b2_V=fd["pp2"].V.astype(np.float32),
            LAT1=LAT1.astype(np.float64),
            LON2=LON2.astype(np.float64),
        )

        info={
            "模型":cfg.PRIMARY_MODEL,
            "随机种子":int(seed),
            "训练样本数":int(len(tr_idx)),
            "验证样本数":int(len(va_idx)),
            "验证开始日期":str(val_start.date()),
            "分支1最佳epoch":int(b1["best_epoch"]),
            "分支2最佳epoch":int(b2["best_epoch"]),
            "训练秒数":float(b1["train_seconds"]+b2["train_seconds"]),
        }
        done.write_text(
            json.dumps(info,indent=2,ensure_ascii=False),
            encoding="utf-8"
        )
        final_rows.append(info)

    pd.DataFrame(final_rows).to_csv(
        DATA_DIR/"14_最终冻结模型训练记录.csv",
        index=False,encoding="utf-8-sig"
    )

    print("最终冻结模型 ->",final_root)


In [ ]:

# ===== 18. R3 最终实验清单 / Handoff =====
handoff={
    "dataset":str(DATASET_PATH),
    "daily_paths":int(len(dates)),
    "date_start":str(dates.min().date()),
    "date_end":str(dates.max().date()),
    "lookback_days":cfg.LOOKBACK_DAYS,
    "lead_days":list(cfg.LEAD_DAYS),
    "valid_anchors":int(len(anchors)),
    "validation_protocol":"strict_causal_blocked_cross_validation",
    "blocked_cv_folds":len(FOLDS),
    "purge_days":cfg.PURGE_DAYS,
    "temporal_leakage_audit":"PASS",
    "eof_variance_gate":"train-only >=99%",
    "reviewer_baselines":[
        "Persistence",
        "Climatology",
        "Recursive AR-EOF",
        "Parameter-matched LSTM",
    ],
    "primary_model":cfg.PRIMARY_MODEL,
    "neural_architectures_for_uncertainty":[
        "Matched-LSTM",
        "EOF-Conv-LSTM",
        "EOF-Conv-GRU",
        "EOF-TCN-LSTM",
        "EOF-TCN-GRU",
    ],
    "neural_seeds":list(cfg.SEEDS),
    "uncertainty":{
        "forecast_error":"hierarchical moving-block bootstrap",
        "network_difference":"paired moving-block bootstrap",
        "axis_detection":"local velocity-ridge localization width + block bootstrap",
        "section_metrics":"monthly moving-block bootstrap if 1993-2024 series supplied",
        "trend":"monthly moving-block bootstrap if 1993-2024 series supplied",
    },
    "longterm_uncertainty_input_ready":bool(longterm_ready),
    "data_directory":str(DATA_DIR),
    "figure_chinese_directory":str(FIG_ZH_DIR),
    "figure_english_directory":str(FIG_EN_DIR),
    "checkpoint_directory":str(CKPT_DIR),
}

(REPORT_DIR/"R3_正式训练实验交接.json").write_text(
    json.dumps(handoff,indent=2,ensure_ascii=False),
    encoding="utf-8"
)

print(json.dumps(handoff,indent=2,ensure_ascii=False))

print("\n核心结果文件：")
for x in [
    "02_严格分块后报折次定义.csv",
    "02_严格时序数据泄漏审计.csv",
    "03_各折训练集EOF配置.csv",
    "04_LSTM与ETL参数量公平性.csv",
    "05_逐样本预测误差.csv",
    "10_主模型与审稿人四类基线比较.csv",
    "13_预测误差Hierarchical_BlockBootstrap_95CI.csv",
    "14_不同网络结构差异_PairedBlockBootstrap_95CI.csv",
    "16_黑潮轴线定位不确定性_断面95CI.csv",
    "18_A至E断面指标_BlockBootstrap_95CI.csv",
    "19_长期趋势_BlockBootstrap_95CI.csv",
]:
    print(" -",DATA_DIR/x)

print("\n中文图表:",FIG_ZH_DIR)
print("英文图表:",FIG_EN_DIR)


## FIX1: tabulate optional dependency removed

This error only affects Markdown summary generation; prior training/results remain valid. Continue from the result-summary cell.


## FIX2 最终判定标准

只有同时看到：

```text
[STRICT TEMPORAL LEAKAGE AUDIT] ALL PASS
[EOF 99% TRAIN-ONLY GATE] ALL PASS
[architecture preflight] ALL PASS
```

才把本轮结果作为 Response to Reviewer 的正式统计。

上一轮 `FINAL_RETRAIN_REVIEWER/` 保留作为 preliminary / sensitivity run；
最终论文只引用 `FINAL_RETRAIN_REVIEWER_FIX2/`。



## R3 运行完成后的判定

投稿用结果至少应同时满足：

```text
[STRICT TEMPORAL LEAKAGE AUDIT] ALL PASS
[EOF 99% TRAIN-ONLY GATE] ALL PASS
[architecture preflight] ALL PASS
```

并检查：

```text
04_LSTM与ETL参数量公平性.csv
```

确认参数量相对差处于可接受范围。

### 关于“多年滚动后报”
当前 daily 数据约为 2023-09 ~ 2025-03，保持 180-day history 和 +90-day lead 后，
没有足够年份构造真正的 multi-year rolling hindcast。
因此正式论文应写：

> strict causal blocked temporal cross-validation / blocked hindcast

而不是把本实验称为 multi-year rolling hindcast。

### 关于长期趋势/断面不确定性
只有当：

```text
长期不确定性输入/1993_2024月尺度监测序列.csv
```

填入原稿真正使用的 1993-2024 月尺度监测序列后，
`18_...` 与 `19_...` 才能作为原稿长期监测结论的不确定性证据。
Notebook 不会用 2023-2025 daily training subset 冒充 1993-2024 长期分析。
